Analysing the drift 
drift_patterns_over_time=[[1, 2],
                         [1, 2],
                         [2, 1]],

In [ ]:
from log_utils.logging import read_logs
import numpy as np

### Get Pre-drift data from logs

In [ ]:
# This code gets the min, max, avg performace stats 
#  - for each simulation timestep,
#  - in all clients,
#  - from the beginning of the simulation unitl the timestep where drift starts

def compute_pre_drift_stats_all_clients(client_log, drift_start):
    """
    Pre-drift stats: for each round t in [0, drift_start),
    compute min/max/avg across ALL clients.
    """
    pre_stats = {}
    for t in range(drift_start):
        losses = [loss for loss, _ in client_log[t]]
        accs   = [acc  for _, acc in client_log[t]]

        pre_stats[t] = {
            "loss_min": min(losses),
            "loss_max": max(losses),
            "loss_avg": sum(losses) / len(losses),
            "acc_min":  min(accs),
            "acc_max":  max(accs),
            "acc_avg":  sum(accs) / len(accs),
        }
    return pre_stats

### Setting up client clusters
##### Arranging the client cluster according to the drift patterns

In [ ]:
# This is a helper function

def extend_clusters_for_final_segment(clusters_per_segment):
    """
    compute_cluster_segmented_stats creates an extra segment [last_drift, T),
    so we reuse the last clustering pattern for that final segment.
    """
    return clusters_per_segment + [clusters_per_segment[-1]]

In [ ]:
def reorder_clusters_by_pattern(
    drift_clustered_client_indices,
    drift_pattern,
    keep_unmentioned="append"  # "append" | "drop"
):
    """
    Reorders clusters at each step according to drift_pattern.

    Supports:
      - subset orders like [1]
      - repeated indices like [1,1]
      - normal permutations like [2,1]

    keep_unmentioned:
      - "append": append clusters not referenced in 'order' at the end (in original order)
      - "drop": only keep clusters referenced in 'order'
    """
    if len(drift_clustered_client_indices) != len(drift_pattern):
        raise ValueError("Length mismatch: clustered indices and pattern must have same #steps.")

    reordered = []

    for step_clusters, order in zip(drift_clustered_client_indices, drift_pattern):
        k = len(step_clusters)

        # Validate each requested cluster index
        for idx in order:
            if not (1 <= idx <= k):
                raise ValueError(f"Invalid cluster index {idx} for step with {k} clusters (valid: 1..{k}).")

        # Reorder (1-based -> 0-based). Allows duplicates.
        reordered_step = [step_clusters[i - 1] for i in order]

        if keep_unmentioned == "append":
            mentioned = set(order)
            # append unmentioned clusters in original order
            for i in range(1, k + 1):
                if i not in mentioned:
                    reordered_step.append(step_clusters[i - 1])
        elif keep_unmentioned == "drop":
            pass
        else:
            raise ValueError("keep_unmentioned must be 'append' or 'drop'.")

        reordered.append(reordered_step)

    return reordered


### Cluster (grouped-clients) stats after drift starts
##### Per cluster seperated stats 

In [ ]:
def per_round_stats_for_clients_in_range(client_log, clients, start_t, end_t):
    """
    client_log[t][client] = (loss, acc)
    Computes per-round min/max/avg over the given 'clients' for t in [start_t, end_t).
    """
    per_round_cluster_stats  = {}
    for t in range(start_t, end_t):
        losses = [client_log[t][c][0] for c in clients]
        accs   = [client_log[t][c][1] for c in clients]

        per_round_cluster_stats[t] = {
            "loss_min": min(losses),
            "loss_max": max(losses),
            "loss_avg": sum(losses) / len(losses),
            "acc_min":  min(accs),
            "acc_max":  max(accs),
            "acc_avg":  sum(accs) / len(accs),
        }
    return per_round_cluster_stats 

In [ ]:
def compute_cluster_segmented_stats(client_log, drift_steps, clusters_per_segment):
    """
    Compute 
        - cluster-wise 
        - per-round statistics, for,
        - each drift segment, 
        - INCLUDING the final segment from the last drift step to the end of the simulation.

    Inputs
    ------
    client_log:
        Nested list where client_log[t][client_id] = (loss, accuracy)
    drift_steps:
        List of drift timestamps, e.g., [20, 33, 35, 45]
        Segments are built as: [20,33), [33,35), [35,45), [45,T)
    clusters_per_segment:
        List aligned with segments. Each element contains clusters for that segment:
          clusters_per_segment[segment_idx][cluster_idx] = [client_ids...]

        IMPORTANT: Must include cluster definitions for the final [last_drift, T) segment as well.

    Output
    ------
    cluster_stats:
        cluster_stats[cluster_idx][(start, end)][t] = {
            "loss_min", "loss_max", "loss_avg",
            "acc_min",  "acc_max",  "acc_avg"
        }
    """
    T = len(client_log)

    # Ensure drift steps are in increasing order so segment construction is correct.
    drift_steps_sorted = sorted(drift_steps)

    if not drift_steps_sorted:
        raise ValueError("drift_steps is empty; cannot build drift segments without at least one drift step.")

    # Build segments between drift points PLUS the final segment to T:
    # e.g., [20,33), [33,35), [35,45), [45,T)
    segment_boundaries = drift_steps_sorted + [T]
    drift_segments = list(zip(segment_boundaries[:-1], segment_boundaries[1:]))

    # clusters_per_segment must provide cluster definitions for EACH segment (including the final one).
    if len(drift_segments) != len(clusters_per_segment):
        raise ValueError(
            f"Mismatch: {len(drift_segments)} drift segments but {len(clusters_per_segment)} cluster definitions."
        )

    # We assume the number of clusters is consistent across segments (e.g., always 3 after appending [8,9]).
    n_clusters = len(clusters_per_segment[0])
    for segment_idx, segment_clusters in enumerate(clusters_per_segment):
        if len(segment_clusters) != n_clusters:
            raise ValueError(
                "All segments must have the same number of clusters "
                "(e.g., after appending [8,9]). "
                f"Segment {segment_idx} has {len(segment_clusters)} clusters, expected {n_clusters}."
            )

    # Prepare output container:
    cluster_stats = {cluster_idx: {} for cluster_idx in range(n_clusters)}

    # For each segment, compute per-round stats for each cluster's client IDs.
    for (segment_start, segment_end), segment_clusters in zip(drift_segments, clusters_per_segment):
        for cluster_idx, client_ids in enumerate(segment_clusters):
            cluster_stats[cluster_idx][(segment_start, segment_end)] = per_round_stats_for_clients_in_range(
                client_log=client_log,
                clients=client_ids,
                start_t=segment_start,
                end_t=segment_end
            )

    return cluster_stats

In [ ]:
# For easy plotting

def cluster_stats_to_plot_series(cluster_stats):
    """
    Converts:
      cluster_stats[k][(start,end)][t] = stats
    into:
      plot_data[k][(start,end)] = {"round": [...], "acc_avg":[...], "loss_avg":[...], ...}
    """
    plot_data = {}

    for k, seg_dict in cluster_stats.items():
        plot_data[k] = {}
        for (start, end), per_round in seg_dict.items():
            rounds = sorted(per_round.keys())
            plot_data[k][(start, end)] = {
                "round": rounds,
                "loss_min": [per_round[r]["loss_min"] for r in rounds],
                "loss_avg": [per_round[r]["loss_avg"] for r in rounds],
                "loss_max": [per_round[r]["loss_max"] for r in rounds],
                "acc_min":  [per_round[r]["acc_min"]  for r in rounds],
                "acc_avg":  [per_round[r]["acc_avg"]  for r in rounds],
                "acc_max":  [per_round[r]["acc_max"]  for r in rounds],
            }
    return plot_data

### Compile log data for performance plots 

In [ ]:
def build_plot_inputs_for_log(client_log, drift_steps, clusters_per_segment_without_final):
    """
    Returns:
      pre_stats: dict[t] -> stats
      post_cluster_plot_stats: output of cluster_stats_to_plot_series(cluster_stats)
    """
    drift_steps_sorted = sorted(drift_steps)
    drift_start = drift_steps_sorted[0]

    pre_stats = compute_pre_drift_stats_all_clients(client_log, drift_start)

    clusters_per_segment = extend_clusters_for_final_segment(clusters_per_segment_without_final)

    cluster_stats = compute_cluster_segmented_stats(
        client_log=client_log,
        drift_steps=drift_steps_sorted,
        clusters_per_segment=clusters_per_segment
    )

    post_cluster_plot_stats = cluster_stats_to_plot_series(cluster_stats)
    return pre_stats, post_cluster_plot_stats

### Plot performance

Description: We have mainly 3 dimentional infomation to show.
   (1) loss and accuracy
   (2) algorithm
   (3) cluster ID or drift ID.
    
So we have the following plot techniques to represent them.
    (1) plot all info in 1 figure
    (2) separate figures for each cluster ID (drift type) but for all algorithms

1) Helper: convert stats dicts into plot arrays

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def stats_dict_to_arrays(stats_by_round, metric_prefix):
    rounds = np.array(sorted(stats_by_round.keys()), dtype=int)
    y_min = np.array([stats_by_round[t][f"{metric_prefix}_min"] for t in rounds], dtype=float)
    y_avg = np.array([stats_by_round[t][f"{metric_prefix}_avg"] for t in rounds], dtype=float)
    y_max = np.array([stats_by_round[t][f"{metric_prefix}_max"] for t in rounds], dtype=float)
    return rounds, y_min, y_avg, y_max

2) Core plotting: one “series” (pre + post clusters) onto a given axis

In [ ]:
def plot_pre_post_drift_continuous_on_axis(
    ax,
    *,
    pre_stats,
    post_cluster_plot_stats,
    metric_prefix,                  # "acc" or "loss"
    base_color,                     # one color per method
    pre_label,
    pre_linestyle="-",              # solid for pre-drift
    pre_linewidth=1.8,
    cluster_linestyles=None,        # dict: {cluster_idx: linestyle}
    cluster_labels=None,            # dict: {cluster_idx: label}
    cluster_linewidth=1.6,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    drift_steps=None,               # list of drift boundaries (vertical lines)
    collect_legend_handles=True,    # whether to collect legend handles for this method
    legend_mode="method",           # "method" | "method+clusters" | "none"
):
    """
    Adds pre-drift + post-drift cluster curves to an existing axis.
    Returns a list of legend handles according to legend_mode.
    """
    if cluster_linestyles is None:
        cluster_linestyles = {}
    if cluster_labels is None:
        cluster_labels = {}

    legend_handles = []

    # -------------------------
    # 1) Pre-drift (all clients)
    # -------------------------
    x_pre, y_pre_min, y_pre_avg, y_pre_max = stats_dict_to_arrays(pre_stats, metric_prefix)

    pre_line, = ax.plot(
        x_pre, y_pre_avg,
        color=base_color,
        linestyle=pre_linestyle,
        linewidth=pre_linewidth,
        label=pre_label
    )

    if collect_legend_handles and legend_mode in ("method", "method+clusters"):
        legend_handles.append(pre_line)

    if show_band:
        ax.fill_between(
            x_pre, y_pre_min, y_pre_max,
            color=base_color,
            alpha=band_alpha,
            linewidth=0
        )

    # Identify the last pre-drift round/value for stitching
    pre_last_round = int(x_pre[-1])
    pre_last_avg = float(y_pre_avg[-1])

    # -------------------------
    # 2) Find first post-drift round
    # -------------------------
    first_post_round = None
    for _, segments_dict in post_cluster_plot_stats.items():
        for (_, _), seg in segments_dict.items():
            if len(seg["round"]) > 0:
                candidate = int(seg["round"][0])
                first_post_round = candidate if first_post_round is None else min(first_post_round, candidate)

    # -------------------------
    # 3) Stitch pre -> post
    # -------------------------
    if stitch and first_post_round is not None and (pre_last_round + 1 == first_post_round):
        for _, segments_dict in post_cluster_plot_stats.items():
            cluster_first_avg = None
            for (_, _), seg in segments_dict.items():
                if len(seg["round"]) == 0:
                    continue
                if int(seg["round"][0]) == first_post_round:
                    cluster_first_avg = float(seg[f"{metric_prefix}_avg"][0])
                    break

            if cluster_first_avg is None:
                continue

            # Stitching line uses the SAME style as pre-drift
            ax.plot(
                [pre_last_round, first_post_round],
                [pre_last_avg, cluster_first_avg],
                color=base_color,
                linestyle=pre_linestyle,
                linewidth=pre_linewidth
            )

    # -------------------------
    # 4) Drift boundary markers
    # -------------------------
    if drift_steps is not None:
        for d in drift_steps:
            ax.axvline(
                x=d,
                color="black",
                linestyle=":",
                linewidth=0.8,
                alpha=0.6,
                zorder=0
            )

    # -------------------------
    # 5) Post-drift (cluster-wise), continuous across segments
    # -------------------------
    for cluster_idx, segments_dict in post_cluster_plot_stats.items():
        linestyle = cluster_linestyles.get(cluster_idx, "--")
        label = cluster_labels.get(cluster_idx, f"{pre_label} | Cluster {cluster_idx}")

        x_all, y_min_all, y_avg_all, y_max_all = [], [], [], []

        for (start, end) in sorted(segments_dict.keys()):
            seg = segments_dict[(start, end)]
            x_seg = np.array(seg["round"], dtype=int)

            y_min_seg = np.array(seg[f"{metric_prefix}_min"], dtype=float)
            y_avg_seg = np.array(seg[f"{metric_prefix}_avg"], dtype=float)
            y_max_seg = np.array(seg[f"{metric_prefix}_max"], dtype=float)

            x_all.append(x_seg)
            y_min_all.append(y_min_seg)
            y_avg_all.append(y_avg_seg)
            y_max_all.append(y_max_seg)

        if not x_all:
            continue

        x = np.concatenate(x_all)
        y_min = np.concatenate(y_min_all)
        y_avg = np.concatenate(y_avg_all)
        y_max = np.concatenate(y_max_all)

        cluster_line, = ax.plot(
            x, y_avg,
            color=base_color,
            linestyle=linestyle,
            linewidth=cluster_linewidth,
            label=label
        )

        # Only include cluster lines in legend when explicitly requested
        if collect_legend_handles and legend_mode == "method+clusters":
            legend_handles.append(cluster_line)

        if show_band:
            ax.fill_between(
                x, y_min, y_max,
                color=base_color,
                alpha=band_alpha,
                linewidth=0
            )

    return legend_handles

3. Plot

In [ ]:
def plot_multiple_methods_pre_post_drift(
    *,
    method_configs,
    drift_steps,
    out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    show_y_label=True,
    filename_suffix="",
    period_annotations=None,
):
    """
    Plots multiple methods (logs) on the same Accuracy and Loss figures and saves both.

    method_configs: list of dicts, each:
      {
        "name": "FedAvg",
        "pre_stats": <dict>,
        "post_cluster_plot_stats": <dict>,
        "color": "red",
        "cluster_linestyles": {0:"--",1:"-.",2:":"},
        "cluster_labels": {0:"...",1:"...",2:"..."}  (optional)
        "show_in_legend": True/False                (optional, default True)
        "legend_mode": "method" | "method+clusters" | "none"  (optional, default "method")
      }
    """
    drift_steps_sorted = sorted(drift_steps)

    fig_acc, ax_acc = plt.subplots()
    ax_acc.set_axisbelow(True)
    ax_acc.grid(True, linestyle=":", color="gray", alpha=0.3)
    fig_loss, ax_loss = plt.subplots()
    ax_loss.set_axisbelow(True)
    ax_loss.grid(True, linestyle=":", color="gray", alpha=0.3)

    acc_handles_all = []
    loss_handles_all = []

    for cfg_idx, cfg in enumerate(method_configs):
        pre_stats = cfg["pre_stats"]
        post_stats = cfg["post_cluster_plot_stats"]

        base_color = cfg.get("color", "black")
        name = cfg.get("name", "method")

        cluster_linestyles = cfg.get("cluster_linestyles", {})
        cluster_labels = cfg.get("cluster_labels", {})

        collect_legend = cfg.get("show_in_legend", True)
        legend_mode = cfg.get("legend_mode", "method")

        # Draw drift lines only once (on the first method) to avoid over-plotting
        drift_lines_for_this_call = drift_steps_sorted if cfg_idx == 0 else None

        acc_handles = plot_pre_post_drift_continuous_on_axis(
            ax_acc,
            pre_stats=pre_stats,
            post_cluster_plot_stats=post_stats,
            metric_prefix="acc",
            base_color=base_color,
            pre_label=f"{name} (pre-drift)",
            pre_linestyle="-",
            cluster_linestyles=cluster_linestyles,
            cluster_labels=cluster_labels,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            drift_steps=drift_lines_for_this_call,
            collect_legend_handles=collect_legend,
            legend_mode=legend_mode,
        )

        loss_handles = plot_pre_post_drift_continuous_on_axis(
            ax_loss,
            pre_stats=pre_stats,
            post_cluster_plot_stats=post_stats,
            metric_prefix="loss",
            base_color=base_color,
            pre_label=f"{name} (pre-drift)",
            pre_linestyle="-",
            pre_linewidth=1.8,
            cluster_linestyles=cluster_linestyles,
            cluster_labels=cluster_labels,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            drift_steps=drift_lines_for_this_call,
            collect_legend_handles=collect_legend,
            legend_mode=legend_mode,
        )

        # Legend collection (no duplicates)
        if collect_legend and acc_handles and loss_handles:
            if legend_mode == "method+clusters":
                acc_handles_all.extend(acc_handles)
                loss_handles_all.extend(loss_handles)
            elif legend_mode == "method":
                acc_handles_all.append(acc_handles[0])
                loss_handles_all.append(loss_handles[0])
            # legend_mode == "none" -> add nothing

    # Optional scenario-period arrows, using the same placement as cluster plots.
    for ax in (ax_acc, ax_loss):
        transform = ax.get_xaxis_transform()
        for left, right, interval_label in (period_annotations or []):
            ax.annotate("", xy=(right, 0.16), xytext=(left, 0.16),
                        xycoords=transform, textcoords=transform,
                        arrowprops={"arrowstyle": "<->", "color": "black",
                                    "linewidth": 1.2, "mutation_scale": 12,
                                    "shrinkA": 0, "shrinkB": 0}, zorder=6)
            ax.text((left + right) / 2, 0.12, interval_label, transform=transform,
                    ha="center", va="top", fontsize=24, color="black", zorder=6)

    # Save both figures once (using your function)
    plt.figure(fig_acc.number)
    configure_and_save_plot(
        _plt=plt,
        plot_formats=("png", "pdf", "pgf"),
        _x_label="Round",
        _y_label="Accuracy" if show_y_label else "",
        _title="",
        _file_path=f"{out_prefix}_accuracy{filename_suffix}",
        _legend_handles=None
    )

    plt.figure(fig_loss.number)
    configure_and_save_plot(
        _plt=plt,
        plot_formats=("png", "pdf", "pgf"),
        _x_label="Round",
        _y_label="Loss" if show_y_label else "",
        _title="",
        _file_path=f"{out_prefix}_loss{filename_suffix}",
        _legend_handles=None
    )

### Save plots

In [ ]:
from plot_utils.plotting import configure_and_save_plot
import constants

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

1.0. Reading drift specifications from the log

In [ ]:
# drift_specs = read_logs('D://publications_related//2026//ICDCS//logs//swap//MNIST//saved_logs_fedavg////drift_specs_log.pkl')  # contains e.g.,
# drift_specs['drift_clustered_client_indices'] =[[[0, 1, 2, 3], [4, 5, 6, 7]],
# [[0, 1], [2, 3, 4, 5, 6, 7]], where [0, 1] -> 1, [2, 3, 4, 5, 6, 7] -> 2
# [[0, 1, 2, 3, 4, 5], [6, 7]]] 
# drift_specs['drift_step_rounds'] = [20, 33, 35, 45]
# drift_specs = read_logs('D://publications_related//2026//ICDCS//logs//swap//CIFAR-10//CIFAR-10_scenario_B_dropfc2fc1//saved_logs_fedavg////drift_specs_log.pkl')  # contains e.g.,
# drift_specs = read_logs('D://publications_related//2026//ICDCS//logs//swap//CIFAR-10//saved_logs_fedavg//drift_specs_log.pkl')
drift_specs = read_logs('D://publications_related//2026//ICDCS//logs//swap//test//CIFAR-10//saved_logs_fedavg//drift_specs_log.pkl')

drift_clustered_client_indices = drift_specs['drift_clustered_client_indices'] 
# drift_pattern=[[1], [1, 2]]  # scenario A
drift_pattern=[[1, 2], [1, 2]]  # scenario B
# drift_pattern=[[1, 2], [1, 2], [2, 1]] # scenario C

#For Sceanio A 
merged_drift_1 = drift_clustered_client_indices[0][0]+drift_clustered_client_indices[0][1]  # merging the groups with drift ID~1 to be 1 list
# drift_2 = []  # for the drift ID~2, that does not consist any clients
drift_2 = merged_drift_1  # for the drift ID~2, that does not consist any clients
drift_clustered_client_indices[0] = [merged_drift_1, drift_2]
drift_clustered_client_indices

reordered_clusters = reorder_clusters_by_pattern(drift_clustered_client_indices, drift_pattern)
print(reordered_clusters)

# This code appends the non drifted client cluster to the reordered_clusters in the previous step

extra_cluster = [8, 9]

reordered_clusters_complete = [
    step_clusters + [extra_cluster]
    for step_clusters in reordered_clusters
]
reordered_clusters_complete

2. Configurations handles to what to diplay on the plot and saving plots

In [ ]:
#For Sceanio A 
drift_clustered_client_indices[0] = drift_clustered_client_indices[0][0]+drift_clustered_client_indices[0][1]  # merging the groups with drift ID~1 to be 1 list
drift_clustered_client_indices[0].append([])  # for the drift ID~2, that does not consist any clients
drift_clustered_client_indices

In [ ]:
# Logs
# fedavg_log  = read_logs('D://publications_related//2026//ICDCS//logs//swap//CIFAR-10//saved_logs_fedavg//client_log.pkl')
# fedex_log   = read_logs('D://publications_related//2026//ICDCS//logs//swap//CIFAR-10//saved_logs_fedex//client_log.pkl')
# oracle_log  = read_logs('D://publications_related//2026//ICDCS//logs//swap//CIFAR-10//saved_logs_oracle//client_log.pkl')

fedavg_log  = read_logs('D://publications_related//2026//ICDCS//logs//swap//test//CIFAR-10//saved_logs_fedavg//client_log.pkl')
fedex_log   = read_logs('D://publications_related//2026//ICDCS//logs//swap//test//CIFAR-10//saved_logs_fedex//client_log.pkl')
oracle_log  = read_logs('D://publications_related//2026//ICDCS//logs//swap//test//CIFAR-10//saved_logs_oracle//client_log.pkl')
# Same scenario and client ordering as the other methods.
fairfeddrift_log  = read_logs('D://publications_related//2026//ICDCS//logs//swap//test//CIFAR-10//saved_logs_fairfeddrift//client_log.pkl')

drift_steps = drift_specs["drift_step_rounds"]  # identical for all
clusters_no_final = reordered_clusters_complete  # identical for all

# Build plot inputs per method
fedavg_pre,  fedavg_post  = build_plot_inputs_for_log(fedavg_log,  drift_steps, clusters_no_final)
fedex_pre,   fedex_post   = build_plot_inputs_for_log(fedex_log,   drift_steps, clusters_no_final)
oracle_pre,  oracle_post  = build_plot_inputs_for_log(oracle_log,  drift_steps, clusters_no_final)
fairfeddrift_pre, fairfeddrift_post = build_plot_inputs_for_log(
    fairfeddrift_log, drift_steps, clusters_no_final
)

# Shared cluster linestyles (same cluster mapping across methods)
cluster_linestyles = {0: "--", 1: "-.", 2: ":"}

method_configs = [
    {
        "name": "FedAvg",
        "pre_stats": fedavg_pre,
        "post_cluster_plot_stats": fedavg_post,
        "color": "red",
        "cluster_linestyles": cluster_linestyles,
        "show_in_legend": True,
        "legend_mode": "method"        # ONE entry only
    },
    {
        "name": "FedEx",
        "pre_stats": fedex_pre,
        "post_cluster_plot_stats": fedex_post,
        "color": "blue",
        "cluster_linestyles": cluster_linestyles,
        "show_in_legend": True,
        "legend_mode": "method"
    },
    {
        "name": "Oracle",
        "pre_stats": oracle_pre,
        "post_cluster_plot_stats": oracle_post,
        "color": "green",
        "cluster_linestyles": cluster_linestyles,
        "show_in_legend": True       # plotted but hidden from legend
    },
    {
        "name": "FairFedDrift",
        "pre_stats": fairfeddrift_pre,
        "post_cluster_plot_stats": fairfeddrift_post,
        "color": "purple",
        "cluster_linestyles": cluster_linestyles,
        "show_in_legend": True,
        "legend_mode": "method"
    }
]

# File save directory and file name
file_name = '1_clients_performance_CIFAR_10'
out_prefix = constants.Paths.PLOT_SAVE_PATH + file_name
# Ensure directory exists if needed:
# import os; os.makedirs("plots/MNIST", exist_ok=True)

plot_multiple_methods_pre_post_drift(
    method_configs=method_configs,
    drift_steps=drift_steps,
    out_prefix=out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True
)

### Create and save custom legends 

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def save_legend_only(
    *,
    handles,
    labels,
    file_path_no_ext,   # e.g., ".../legend_only"
    ncol=None,
    fontsize=8,
    handlelength=2.5,
    columnspacing=1.5
):
    """
    Saves a legend as a standalone figure (PNG + PDF + PGF).
    """
    if ncol is None:
        ncol = len(handles)

    fig = plt.figure(figsize=(max(6, 0.9 * len(handles)), 0.8))  # wide & short
    fig.legend(
        handles,
        labels,
        loc="center",
        ncol=ncol,
        frameon=False,
        fontsize=fontsize,
        handlelength=handlelength,
        columnspacing=columnspacing,
    )
    plt.axis("off")

    # Preserve the legend crop while using the repository's PGF export path.
    with plt.rc_context({"savefig.bbox": "tight", "savefig.pad_inches": 0.05}):
        plt.figure(fig.number)
        configure_and_save_plot(
            _plt=plt,
            _x_label="",
            _y_label="",
            _title="",
            _file_path=file_path_no_ext,
            plot_formats=("png", "pdf", "pgf"),
        )

In [ ]:
def build_custom_legend_handles():
    # ---- method styles (example; adjust to your actual plot styles) ----
    method_items = [
        ("FedAvg", "red",   "-"),    # (label, color, linestyle)
        ("PaLA",  "blue",  "-"),
        ("Oracle", "green", "-"),
        ("FairFedDrift", "purple", "-"),
    ]

    # ---- drift/cluster styles in black (example; adjust to your meaning) ----
    drift_items = [
        ("Drift 1",  "black", "--"),
        ("Drift 2",  "black", "-."),
        ("No drift", "black", ":"),   # or "-" if you prefer solid for no-drift
    ]

    handles = []
    labels = []

    for label, color, ls in method_items + drift_items:
        handles.append(Line2D([0], [0], color=color, linestyle=ls, linewidth=1.8))
        labels.append(label)

    return handles, labels

In [ ]:
# Save legend
handles, labels = build_custom_legend_handles()

legend_out = constants.Paths.PLOT_SAVE_PATH + "legend_only"
save_legend_only(
    handles=handles,
    labels=labels,
    file_path_no_ext=legend_out,
    ncol=len(handles)   # all in one row
)

### Plot seperate graphs based on drift types

1. Detailed axes plot information

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_pre_post_single_cluster_on_axis(
    ax,
    *,
    pre_stats,
    post_cluster_plot_stats,
    cluster_idx,
    metric_prefix,              # "acc" or "loss"
    base_color,
    method_label,
    pre_linestyle="-",
    pre_linewidth=1.8,
    post_linestyle="--",        # <-- drift-type controlled (passed in)
    post_linewidth=1.6,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    drift_steps=None,
    collect_legend=True,
    smooth_window=1,            # optional: keep if you use smoothing elsewhere
    smooth_mode="moving_avg",   # optional
):
    """
    Plot one method on a given axis:
      - pre-drift (all clients) avg + band
      - post-drift ONLY for one cluster_idx avg + band
      - stitching line from last pre point to first post point (for this cluster)
      - optional drift boundary lines

    Returns: legend handles (typically one handle for the method).
    """

    legend_handles = []

    # ---- Pre-drift ----
    x_pre, y_pre_min, y_pre_avg, y_pre_max = stats_dict_to_arrays(pre_stats, metric_prefix)

    # Average line (pre)
    pre_line, = ax.plot(
        x_pre, y_pre_avg,
        color=base_color,
        linestyle=pre_linestyle,
        linewidth=pre_linewidth,
        label=method_label
    )
    if collect_legend:
        legend_handles.append(pre_line)

    # Band (pre)
    if show_band:
        ax.fill_between(
            x_pre, y_pre_min, y_pre_max,
            color=base_color,
            alpha=band_alpha,
            linewidth=0
        )

    pre_last_round = int(x_pre[-1])
    pre_last_avg = float(y_pre_avg[-1])

    # ---- Drift boundary markers ----
    if drift_steps is not None:
        for d in drift_steps:
            ax.axvline(
                x=d,
                color="black",
                linestyle=":",
                linewidth=0.8,
                alpha=0.6,
                zorder=0
            )

    # ---- Post-drift: ONLY selected cluster ----
    if cluster_idx not in post_cluster_plot_stats:
        return legend_handles

    segments_dict = post_cluster_plot_stats[cluster_idx]
    if not segments_dict:
        return legend_handles

    # Concatenate segments in time order (continuous across segments)
    x_all, y_min_all, y_avg_all, y_max_all = [], [], [], []

    for (start, end) in sorted(segments_dict.keys()):
        seg = segments_dict[(start, end)]
        if len(seg["round"]) == 0:
            continue

        x_seg = np.array(seg["round"], dtype=int)
        y_min_seg = np.array(seg[f"{metric_prefix}_min"], dtype=float)
        y_avg_seg = np.array(seg[f"{metric_prefix}_avg"], dtype=float)
        y_max_seg = np.array(seg[f"{metric_prefix}_max"], dtype=float)

        x_all.append(x_seg)
        y_min_all.append(y_min_seg)
        y_avg_all.append(y_avg_seg)
        y_max_all.append(y_max_seg)

    if not x_all:
        return legend_handles

    x_post = np.concatenate(x_all)
    y_post_min = np.concatenate(y_min_all)
    y_post_avg = np.concatenate(y_avg_all)
    y_post_max = np.concatenate(y_max_all)

    # ---- Stitch pre -> post for this cluster ----
    if stitch and len(x_post) > 0:
        first_post_round = int(x_post[0])
        first_post_avg = float(y_post_avg[0])
        if pre_last_round + 1 == first_post_round:
            ax.plot(
                [pre_last_round, first_post_round],
                [pre_last_avg, first_post_avg],
                color=base_color,
                linestyle=pre_linestyle,
                linewidth=pre_linewidth
            )

    # ---- Post line (cluster) ----
    ax.plot(
        x_post, y_post_avg,
        color=base_color,
        linestyle=post_linestyle,   # <-- drift-type linestyle comes from caller
        linewidth=post_linewidth
    )

    # Band (post)
    if show_band:
        ax.fill_between(
            x_post, y_post_min, y_post_max,
            color=base_color,
            alpha=band_alpha,
            linewidth=0
        )

    return legend_handles

2. Config handling and saving plots

In [ ]:
def plot_one_cluster_across_methods(
    *,
    method_configs,
    cluster_idx,
    metric_prefix,           # "acc" or "loss"
    drift_steps,
    out_file_path,           # full prefix (no extension)
    post_linestyle="--",     # <-- drift-type linestyle controlled from execution block
    title="",
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    smooth_window=1,         # optional
    smooth_mode="moving_avg", # optional
    t1_range=None,
    period_annotations=None,
    show_y_label=True,
    filename_suffix="",
    pre_linewidth=1.8,
    post_linewidth=1.6,
):
    """
    Create one figure for ONE cluster (cluster_idx) and ONE metric (acc/loss),
    overlaying all methods (FedAvg/FedEx/Oracle/FairFedDrift) with different colors.
    """

    fig, ax = plt.subplots()

    drift_steps_sorted = sorted(drift_steps)
    legend_handles = []

    for cfg in method_configs:
        handles = plot_pre_post_single_cluster_on_axis(
            ax,
            pre_stats=cfg["pre_stats"],
            post_cluster_plot_stats=cfg["post_cluster_plot_stats"],
            cluster_idx=cluster_idx,
            metric_prefix=metric_prefix,
            base_color=cfg.get("color", "black"),
            method_label=cfg.get("name", "method"),
            pre_linestyle="-",
            pre_linewidth=pre_linewidth,
            post_linestyle=post_linestyle,   # <-- same linestyle for all methods in this drift-type plot
            post_linewidth=post_linewidth,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            drift_steps=drift_steps_sorted,
            collect_legend=True,
            smooth_window=smooth_window,
            smooth_mode=smooth_mode
        )
        legend_handles.extend(handles)

    if t1_range is not None:
        left, right = t1_range
        transform = ax.get_xaxis_transform()
        ax.annotate("", xy=(right, 0.16), xytext=(left, 0.16),
                    xycoords=transform, textcoords=transform,
                    arrowprops={"arrowstyle": "<->", "color": "black",
                                "linewidth": 1.2, "mutation_scale": 12,
                                "shrinkA": 0, "shrinkB": 0}, zorder=6)
        ax.text((left + right) / 2, 0.12, r"$T_1$", transform=transform,
                ha="center", va="top", fontsize=24, color="black", zorder=6)

    for left, right, interval_label in (period_annotations or []):
        transform = ax.get_xaxis_transform()
        ax.annotate("", xy=(right, 0.16), xytext=(left, 0.16),
                    xycoords=transform, textcoords=transform,
                    arrowprops={"arrowstyle": "<->", "color": "black",
                                "linewidth": 1.2, "mutation_scale": 12,
                                "shrinkA": 0, "shrinkB": 0}, zorder=6)
        ax.text((left + right) / 2, 0.12, interval_label, transform=transform,
                ha="center", va="top", fontsize=24, color="black", zorder=6)

    plt.figure(fig.number)
    configure_and_save_plot(
        _plt=plt,
        plot_formats=("png", "pdf", "pgf"),
        _x_label="Round",
        _y_label=("Accuracy" if metric_prefix == "acc" else "Loss") if show_y_label else "",
        _title=title,
        _file_path=out_file_path + filename_suffix,
        _legend_handles=None
    )


3. Plotting function

In [ ]:
def plot_all_clusters_separate(
    *,
    method_configs,
    drift_steps,
    cluster_map,              # e.g., {0:"drift_1", 1:"drift_2", 2:"no_drift"}
    drift_type_linestyles,    # e.g., {"drift_1":"--", "drift_2":"-.", "no_drift":":"}
    out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    smooth_window=1,
    smooth_mode="moving_avg",
    t1_range=None,
    period_annotations=None,
    scenario=None,
    show_y_label=True,
    filename_suffix="",
):
    """
    Generates separate figures per drift type / cluster:
      - Accuracy for each drift type
      - Loss for each drift type

    Drift-type linestyles are controlled from the execution block via drift_type_linestyles.
    """

    for cluster_idx, drift_name in cluster_map.items():
        cluster_t1_range = None if drift_name == "drift_2" else t1_range
        post_ls = drift_type_linestyles.get(drift_name, "--")  # fallback
        if drift_name == "drift_1" and scenario in ("A", "B", "C"):
            file_prefix = "{}_{}_drift_1".format(out_prefix, scenario)
        else:
            file_prefix = "{}_{}".format(out_prefix, drift_name)

        # Accuracy plot for this drift type
        plot_one_cluster_across_methods(
            method_configs=method_configs,
            cluster_idx=cluster_idx,
            metric_prefix="acc",
            drift_steps=drift_steps,
            out_file_path=f"{file_prefix}_accuracy",
            post_linestyle=post_ls,
            title="",
            t1_range=cluster_t1_range,
            period_annotations=period_annotations,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            smooth_window=smooth_window,
            smooth_mode=smooth_mode,
            show_y_label=show_y_label,
            filename_suffix=filename_suffix,
        )

        # Loss plot for this drift type
        plot_one_cluster_across_methods(
            method_configs=method_configs,
            cluster_idx=cluster_idx,
            metric_prefix="loss",
            drift_steps=drift_steps,
            out_file_path=f"{file_prefix}_loss",
            post_linestyle=post_ls,
            title="",
            t1_range=cluster_t1_range,
            period_annotations=period_annotations,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            smooth_window=smooth_window,
            smooth_mode=smooth_mode,
            show_y_label=show_y_label,
            filename_suffix=filename_suffix,
        )


4. Execute

In [ ]:
cluster_map = {0: "drift_1", 1: "drift_2", 2: "no_drift"}

drift_type_linestyles = {
    "drift_1": "--",
    "drift_2": "-.",
    "no_drift": ":"
}

file_name = "3_clusterwise_comparison_CIFAR-10"
out_prefix = constants.Paths.PLOT_SAVE_PATH + file_name

plot_all_clusters_separate(
    method_configs=method_configs,   # FedAvg/FedEx/Oracle/FairFedDrift
    drift_steps=drift_steps,
    cluster_map=cluster_map,
    drift_type_linestyles=drift_type_linestyles,
    out_prefix=out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    smooth_window=9,                 # optional smoothing
    smooth_mode="moving_avg"
)

### Focus/Opacity plotting functions

1. Helper functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def _in_any_range(x, ranges):
    """Return boolean mask where x is inside any (lo, hi) interval."""
    if ranges is None:
        return np.ones_like(x, dtype=bool)
    mask = np.zeros_like(x, dtype=bool)
    for lo, hi in ranges:
        mask |= (x >= lo) & (x <= hi)
    return mask

In [ ]:
def _plot_line_with_focus_alpha(
    ax, x, y, *, color, linestyle, linewidth, focus_ranges, alpha_focus=1.0, alpha_other=0.12, label=None
):
    """
    Plot a single line with full opacity inside focus_ranges and faded elsewhere.
    Returns the first Line2D handle (for legend usage).
    """
    x = np.asarray(x)
    y = np.asarray(y, dtype=float)

    focus = _in_any_range(x, focus_ranges)
    handle = None

    # Find contiguous runs of focus True/False and plot them separately
    start = 0
    for i in range(1, len(x) + 1):
        if i == len(x) or focus[i] != focus[i - 1]:
            seg_x = x[start:i]
            seg_y = y[start:i]
            seg_alpha = alpha_focus if focus[i - 1] else alpha_other

            # only attach label to the first plotted segment (for clean legends)
            seg_label = label if handle is None else None

            line, = ax.plot(seg_x, seg_y, color=color, linestyle=linestyle, linewidth=linewidth,
                            alpha=seg_alpha, label=seg_label)
            if handle is None:
                handle = line
            start = i

    return handle

In [ ]:
def _fill_between_with_focus_alpha(
    ax, x, y1, y2, *, color, focus_ranges, alpha_focus=0.25, alpha_other=0.05
):
    """
    Fill band with higher alpha inside focus_ranges and lower alpha elsewhere.
    """
    x = np.asarray(x)
    y1 = np.asarray(y1, dtype=float)
    y2 = np.asarray(y2, dtype=float)

    focus = _in_any_range(x, focus_ranges)

    start = 0
    for i in range(1, len(x) + 1):
        if i == len(x) or focus[i] != focus[i - 1]:
            seg_x = x[start:i]
            seg_y1 = y1[start:i]
            seg_y2 = y2[start:i]
            seg_alpha = alpha_focus if focus[i - 1] else alpha_other

            ax.fill_between(seg_x, seg_y1, seg_y2, color=color, alpha=seg_alpha, linewidth=0)
            start = i


In [ ]:
def plot_pre_post_single_cluster_on_axis_focus(
    ax,
    *,
    pre_stats,
    post_cluster_plot_stats,
    cluster_idx,
    metric_prefix,              # "acc" or "loss"
    base_color,
    method_label,
    pre_linestyle="-",
    pre_linewidth=1.8,
    post_linestyle="--",        # drift-type controlled
    post_linewidth=1.6,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    drift_steps=None,
    collect_legend=True,
    focus_x_ranges=None,       
    faded_alpha=0.12,          
):
    """
    Same as your original per-cluster plot, but with focus/opacity control:
      - Inside focus_x_ranges: fully opaque
      - Outside focus_x_ranges: faded
    """

    legend_handles = []

    # ---- Pre-drift ----
    x_pre, y_pre_min, y_pre_avg, y_pre_max = stats_dict_to_arrays(pre_stats, metric_prefix)

    # pre avg line with focus fade
    pre_handle = _plot_line_with_focus_alpha(
        ax, x_pre, y_pre_avg,
        color=base_color,
        linestyle=pre_linestyle,
        linewidth=pre_linewidth,
        focus_ranges=focus_x_ranges,
        alpha_focus=1.0,
        alpha_other=faded_alpha,
        label=method_label
    )

    if collect_legend and pre_handle is not None:
        legend_handles.append(pre_handle)

    # pre band with focus fade
    if show_band:
        _fill_between_with_focus_alpha(
            ax, x_pre, y_pre_min, y_pre_max,
            color=base_color,
            focus_ranges=focus_x_ranges,
            alpha_focus=band_alpha,
            alpha_other=min(0.08, band_alpha * 0.3)
        )

    pre_last_round = int(x_pre[-1])
    pre_last_avg = float(y_pre_avg[-1])

    # ---- Drift boundary markers ----
    if drift_steps is not None:
        for d in drift_steps:
            ax.axvline(x=d, color="black", linestyle=":", linewidth=0.8, alpha=0.6, zorder=0)

    # ---- Post-drift: ONLY selected cluster ----
    if cluster_idx not in post_cluster_plot_stats:
        return legend_handles

    segments_dict = post_cluster_plot_stats[cluster_idx]
    if not segments_dict:
        return legend_handles

    x_all, y_min_all, y_avg_all, y_max_all = [], [], [], []
    for (start, end) in sorted(segments_dict.keys()):
        seg = segments_dict[(start, end)]
        if len(seg["round"]) == 0:
            continue

        x_seg = np.array(seg["round"], dtype=int)
        y_min_seg = np.array(seg[f"{metric_prefix}_min"], dtype=float)
        y_avg_seg = np.array(seg[f"{metric_prefix}_avg"], dtype=float)
        y_max_seg = np.array(seg[f"{metric_prefix}_max"], dtype=float)

        x_all.append(x_seg)
        y_min_all.append(y_min_seg)
        y_avg_all.append(y_avg_seg)
        y_max_all.append(y_max_seg)

    if not x_all:
        return legend_handles

    x_post = np.concatenate(x_all)
    y_post_min = np.concatenate(y_min_all)
    y_post_avg = np.concatenate(y_avg_all)
    y_post_max = np.concatenate(y_max_all)

    # ---- Stitch pre -> post for this cluster ----
    if stitch and len(x_post) > 0:
        first_post_round = int(x_post[0])
        first_post_avg = float(y_post_avg[0])
        if pre_last_round + 1 == first_post_round:
            # stitching uses pre style, and should respect focus opacity too
            _plot_line_with_focus_alpha(
                ax,
                [pre_last_round, first_post_round],
                [pre_last_avg, first_post_avg],
                color=base_color,
                linestyle=pre_linestyle,
                linewidth=pre_linewidth,
                focus_ranges=focus_x_ranges,
                alpha_focus=1.0,
                alpha_other=faded_alpha,
                label=None
            )

    # ---- Post avg line with focus fade ----
    _plot_line_with_focus_alpha(
        ax, x_post, y_post_avg,
        color=base_color,
        linestyle=post_linestyle,
        linewidth=post_linewidth,
        focus_ranges=focus_x_ranges,
        alpha_focus=1.0,
        alpha_other=faded_alpha,
        label=None
    )

    # ---- Post band with focus fade ----
    if show_band:
        _fill_between_with_focus_alpha(
            ax, x_post, y_post_min, y_post_max,
            color=base_color,
            focus_ranges=focus_x_ranges,
            alpha_focus=band_alpha,
            alpha_other=min(0.08, band_alpha * 0.3)
        )

    return legend_handles

In [ ]:
def plot_one_cluster_across_methods_focus(
    *,
    method_configs,
    cluster_idx,
    metric_prefix,           # "acc" or "loss"
    drift_steps,
    out_file_path,           # prefix without extension
    post_linestyle="--",     # drift-type linestyle
    title="",
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    show_legend=True,
    axis_label_fontsize=14,
    tick_fontsize=12,
    legend_fontsize=10,
    focus_x_ranges=None,    
    faded_alpha=0.12,        
    t1_range=None,
):
    fig, ax = plt.subplots()

    drift_steps_sorted = sorted(drift_steps)
    legend_handles = []

    for cfg in method_configs:
        handles = plot_pre_post_single_cluster_on_axis_focus(
            ax,
            pre_stats=cfg["pre_stats"],
            post_cluster_plot_stats=cfg["post_cluster_plot_stats"],
            cluster_idx=cluster_idx,
            metric_prefix=metric_prefix,
            base_color=cfg.get("color", "black"),
            method_label=cfg.get("name", "method"),
            pre_linestyle="-",
            pre_linewidth=1.8,
            post_linestyle=post_linestyle,
            post_linewidth=1.6,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            drift_steps=drift_steps_sorted,
            collect_legend=show_legend,
            focus_x_ranges=focus_x_ranges,
            faded_alpha=faded_alpha,
        )
        legend_handles.extend(handles)

    if t1_range is not None:
        left, right = t1_range
        transform = ax.get_xaxis_transform()
        ax.annotate("", xy=(right, 0.16), xytext=(left, 0.16),
                    xycoords=transform, textcoords=transform,
                    arrowprops={"arrowstyle": "<->", "color": "black",
                                "linewidth": 1.2, "mutation_scale": 12,
                                "shrinkA": 0, "shrinkB": 0}, zorder=6)
        ax.text((left + right) / 2, 0.12, r"$T_1$", transform=transform,
                ha="center", va="top", fontsize=24, color="black", zorder=6)

    # legend control
    legend_arg = legend_handles if show_legend else None

    plt.figure(fig.number)
    configure_and_save_plot(
        _plt=plt,
        plot_formats=("png", "pdf", "pgf"),
        _x_label="Round",
        _y_label="Accuracy" if metric_prefix == "acc" else "Loss",
        _title=title,
        _file_path=out_file_path,
        _legend_handles=None,
        # axis_label_fontsize=axis_label_fontsize,
        # tick_fontsize=tick_fontsize,
        # legend_fontsize=legend_fontsize
    )


In [ ]:
def plot_all_clusters_separate_focus(
    *,
    method_configs,
    drift_steps,
    cluster_map,              # e.g. {0:"drift_1", 1:"drift_2", 2:"no_drift"}
    drift_type_linestyles,    # e.g. {"drift_1":"--","drift_2":"-.", "no_drift":":"}
    out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    show_legend=True,
    axis_label_fontsize=14,
    tick_fontsize=12,
    legend_fontsize=10,
    focus_x_ranges=None,    
    faded_alpha=0.12,    
    t1_range=None,
):
    for cluster_idx, drift_name in cluster_map.items():
        post_ls = drift_type_linestyles.get(drift_name, "--")

        # Accuracy
        plot_one_cluster_across_methods_focus(
            method_configs=method_configs,
            cluster_idx=cluster_idx,
            metric_prefix="acc",
            drift_steps=drift_steps,
            out_file_path=f"{out_prefix}_{drift_name}_accuracy_focus",
            post_linestyle=post_ls,
            title="",
            t1_range=t1_range,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            show_legend=show_legend,
            axis_label_fontsize=axis_label_fontsize,
            tick_fontsize=tick_fontsize,
            legend_fontsize=legend_fontsize,
            focus_x_ranges=focus_x_ranges,
            faded_alpha=faded_alpha
        )

        # Loss
        plot_one_cluster_across_methods_focus(
            method_configs=method_configs,
            cluster_idx=cluster_idx,
            metric_prefix="loss",
            drift_steps=drift_steps,
            out_file_path=f"{out_prefix}_{drift_name}_loss_focus",
            post_linestyle=post_ls,
            title="",
            t1_range=t1_range,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            show_legend=show_legend,
            axis_label_fontsize=axis_label_fontsize,
            tick_fontsize=tick_fontsize,
            legend_fontsize=legend_fontsize,
            focus_x_ranges=focus_x_ranges,
            faded_alpha=faded_alpha
        )

2. Execution block

In [ ]:
# ---- Focus range handle: only these x-ranges are fully opaque ----
focus_x_ranges = [
    (0, 40),     # show warm-up
    # (40, 65),    # show around first drift
    (65, 100),    # show around second drift
]

# How faded should the rest be?
faded_alpha = 0.3 # 0.05 very faint, 0.15 moderate

file_name = "3_clusterwise_comparison_CIFAR-10_A_shaded"
out_prefix = constants.Paths.PLOT_SAVE_PATH + file_name

plot_all_clusters_separate_focus(
    method_configs=method_configs,
    drift_steps=drift_steps,
    cluster_map=cluster_map,
    drift_type_linestyles=drift_type_linestyles,
    out_prefix=out_prefix + "_FOCUS",
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    show_legend=False,           # often better when focusing
    axis_label_fontsize=18,
    tick_fontsize=16,
    legend_fontsize=12,
    focus_x_ranges=focus_x_ranges,
    faded_alpha=faded_alpha,
    t1_range=drift_steps[:2] if len(drift_steps) > 1 else None
)

## CIFAR-10 rotation logs: raw and smoothed plots

This independent block reads the Scenario B rotation logs, plots all five methods, and writes raw and nine-round-smoothed figures to separate figure folders.

In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from log_utils.logging import read_logs

ROTATION_LOG_ROOT = Path("D:/publications_related/2026/ICDCS/logs/swap/seed42/B/CIFAR_rot/CIFAR-10")
ROTATION_FIGURE_ROOT = Path("D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_rotation")
ROTATION_SMOOTH_ROOT = ROTATION_FIGURE_ROOT / "smooth"
ROTATION_SMOOTH_WINDOW = 9

plt.rcParams.update({"savefig.bbox": "tight", "savefig.pad_inches": 0.1})
plt.rcParams.update({"font.size": 20, "axes.labelsize": 20,
                     "xtick.labelsize": 20, "ytick.labelsize": 20})

def configure_and_save_plot(_plt, _x_label, _y_label, _title, _file_path,
                            _legend_handles=None, plot_formats=("png", "pdf", "pgf")):
    _plt.xlabel(_x_label)
    _plt.ylabel(_y_label)
    if _legend_handles is not None:
        _plt.legend(handles=_legend_handles, loc="best", fontsize=10)
    os.makedirs(os.path.dirname(str(_file_path)), exist_ok=True)
    if "png" in plot_formats:
        _plt.savefig(str(_file_path) + ".png", dpi=300)
    if "pdf" in plot_formats:
        _plt.savefig(str(_file_path) + ".pdf", format="pdf")
    if "pgf" in plot_formats:
        try:
            _plt.savefig(str(_file_path) + ".pgf", format="pgf")
        except Exception as exc:
            warnings.warn("PGF export failed for {}: {}".format(_file_path, exc), RuntimeWarning)
    _plt.close()

def annotate_drift_intervals(ax, boundaries, labels=("2", "3"), arrow_y=0.16, label_y=0.12):
    transform = ax.get_xaxis_transform()
    for left, right, label in zip(boundaries[:-1], boundaries[1:], labels):
        ax.annotate("", xy=(right, arrow_y), xytext=(left, arrow_y),
                    xycoords=transform, textcoords=transform,
                    arrowprops=dict(arrowstyle="<->", color="black", linewidth=1.2,
                                    mutation_scale=12, shrinkA=0, shrinkB=0), zorder=6)
        ax.text((left + right) / 2, label_y, rf"$T_{{{label}}}$", transform=transform,
                ha="center", va="top", fontsize=24, color="black", zorder=6)

def annotate_scenario_intervals(ax, boundaries, pattern):
    if pattern == [[1, 2], [1, 2]]:
        annotate_drift_intervals(ax, boundaries, labels=("2", "3"))

def smooth_client_log(client_log, window=ROTATION_SMOOTH_WINDOW):
    """Centered moving average per client, for loss and accuracy."""
    if window < 1:
        raise ValueError("Smoothing window must be at least one round.")
    if window == 1:
        return client_log
    result = []
    left = (window - 1) // 2
    right = window // 2
    for round_idx in range(len(client_log)):
        start = max(0, round_idx - left)
        end = min(len(client_log), round_idx + right + 1)
        row = []
        for client_idx in range(len(client_log[0])):
            losses = [client_log[t][client_idx][0] for t in range(start, end)]
            accuracies = [client_log[t][client_idx][1] for t in range(start, end)]
            row.append((float(np.mean(losses)), float(np.mean(accuracies))))
        result.append(row)
    return result

rotation_drift_specs = read_logs(str(ROTATION_LOG_ROOT / "saved_logs_fedavg" / "drift_specs_log.pkl"))
drift_steps = rotation_drift_specs["drift_step_rounds"]
rotation_period_annotations = [
    (drift_steps[0], drift_steps[1], r"$T_{2}$"),
    (drift_steps[1], drift_steps[2], r"$T_{3}$"),
]
drift_pattern = [[1, 2], [1, 2]]
drift_clustered_client_indices = rotation_drift_specs["drift_clustered_client_indices"]
reordered_clusters = reorder_clusters_by_pattern(drift_clustered_client_indices, drift_pattern)
clusters_no_final = [groups + [[8, 9]] for groups in reordered_clusters]

rotation_method_specs = [
    ("FedAvg", "saved_logs_fedavg", "red"),
    ("Oracle", "saved_logs_oracle", "green"),
    ("FairFedDrift", "saved_logs_fairfeddrift", "darkviolet"),
    ("FedBABU", "saved_logs_fedbabu", "darkorange"),
    ("PaLA", "saved_logs_fedex", "blue"),
]
cluster_linestyles = {0: "--", 1: "-.", 2: ":"}

rotation_raw_logs = {}
rotation_raw_configs = []
rotation_smooth_configs = []
for method_name, folder_name, color in rotation_method_specs:
    log_path = ROTATION_LOG_ROOT / folder_name / "client_log.pkl"
    if not log_path.is_file():
        raise FileNotFoundError("Missing method log: {}".format(log_path))
    raw_log = read_logs(str(log_path))
    rotation_raw_logs[method_name] = raw_log
    pre_stats, post_stats = build_plot_inputs_for_log(raw_log, drift_steps, clusters_no_final)
    rotation_raw_configs.append({
        "name": method_name, "pre_stats": pre_stats,
        "post_cluster_plot_stats": post_stats, "color": color,
        "cluster_linestyles": cluster_linestyles, "show_in_legend": True,
        "legend_mode": "method",
    })
    smooth_pre, smooth_post = build_plot_inputs_for_log(
        smooth_client_log(raw_log), drift_steps, clusters_no_final
    )
    rotation_smooth_configs.append({
        "name": method_name, "pre_stats": smooth_pre,
        "post_cluster_plot_stats": smooth_post, "color": color,
        "cluster_linestyles": cluster_linestyles, "show_in_legend": True,
        "legend_mode": "method",
    })

rotation_cluster_map = {0: "drift_1", 1: "drift_2", 2: "no_drift"}
rotation_drift_styles = {"drift_1": "--", "drift_2": "-.", "no_drift": ":"}
print("Rotation logs:", [spec[0] for spec in rotation_method_specs])
print("Drift rounds:", drift_steps, "; smoothing window:", ROTATION_SMOOTH_WINDOW)


In [ ]:
# Combined accuracy/loss figures, then drift-specific figures for both versions.
rotation_raw_prefix = str(ROTATION_FIGURE_ROOT / "1_clients_performance_CIFAR-10_rotation")
rotation_smooth_prefix = str(ROTATION_SMOOTH_ROOT / "1_clients_performance_CIFAR-10_rotation")

plot_multiple_methods_pre_post_drift(
    method_configs=rotation_raw_configs, drift_steps=drift_steps,
    out_prefix=rotation_raw_prefix, band_alpha=0.25, show_band=True, stitch=True, period_annotations=rotation_period_annotations,
)
plot_multiple_methods_pre_post_drift(
    method_configs=rotation_smooth_configs, drift_steps=drift_steps,
    out_prefix=rotation_smooth_prefix, band_alpha=0.25, show_band=True, stitch=True, period_annotations=rotation_period_annotations,
)

plot_all_clusters_separate(
    method_configs=rotation_raw_configs, drift_steps=drift_steps,
    cluster_map=rotation_cluster_map, drift_type_linestyles=rotation_drift_styles,
    out_prefix=str(ROTATION_FIGURE_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=True, stitch=True, period_annotations=rotation_period_annotations,
)
plot_all_clusters_separate(
    method_configs=rotation_smooth_configs, drift_steps=drift_steps,
    cluster_map=rotation_cluster_map, drift_type_linestyles=rotation_drift_styles,
    out_prefix=str(ROTATION_SMOOTH_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=True, stitch=True, period_annotations=rotation_period_annotations,
)

def save_rotation_legend(configs, file_prefix):
    handles = [Line2D([0], [0], color=cfg["color"], linewidth=3.6, alpha=0.6,
                      label=cfg["name"]) for cfg in configs]
    fig = plt.figure(figsize=(max(6, len(handles) * 1.4), 0.8))
    fig.legend(handles=handles, labels=[cfg["name"] for cfg in configs], loc="center",
               ncol=len(handles), frameon=False, fontsize=16, handlelength=2.5,
               columnspacing=1.5)
    os.makedirs(os.path.dirname(file_prefix), exist_ok=True)
    fig.savefig(file_prefix + ".png", dpi=300, bbox_inches="tight", pad_inches=0.1)
    fig.savefig(file_prefix + ".pdf", bbox_inches="tight", pad_inches=0.1)
    try:
        fig.savefig(file_prefix + ".pgf", format="pgf", bbox_inches="tight", pad_inches=0.1)
    except Exception as exc:
        warnings.warn("PGF legend export failed: {}".format(exc), RuntimeWarning)
    plt.close(fig)

save_rotation_legend(rotation_raw_configs, str(ROTATION_FIGURE_ROOT / "CIFAR-10_rotation_legend"))
save_rotation_legend(rotation_smooth_configs, str(ROTATION_SMOOTH_ROOT / "CIFAR-10_rotation_legend"))


In [ ]:
# Batch plots from seed42 Scenarios A, B, and C.
from pathlib import Path
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from log_utils.logging import read_logs

BATCH_LOG_ROOT = Path("D:/publications_related/2026/ICDCS/logs/swap/seed42")
BATCH_FIGURE_ROOT = Path("D:/publications_related/2026/ICDCS/figures/new/seed42/l3")
BATCH_DATASET = "CIFAR-10"
BATCH_METHODS = [
    ("FedAvg", "saved_logs_fedavg", "red"),
    ("Oracle", "saved_logs_oracle", "green"),
    ("FairFedDrift", "saved_logs_fairfeddrift", "darkviolet"),
    ("FedBABU", "saved_logs_fedbabu", "darkorange"),
    ("PaLA", "saved_logs_fedex", "blue"),
]
BATCH_PATTERNS = {
    "A": [[1], [1, 2]],
    "B": [[1, 2], [1, 2]],
    "C": [[1, 2], [1, 2], [2, 1]],
}

plt.rcParams.update({"savefig.bbox": "tight", "savefig.pad_inches": 0.1,
                     "font.size": 20, "axes.labelsize": 20,
                     "xtick.labelsize": 20, "ytick.labelsize": 20})

def configure_and_save_plot(_plt, _x_label, _y_label, _title, _file_path,
                            _legend_handles=None, plot_formats=("png", "pdf", "pgf")):
    """Format and save a notebook plot with an external legend and dotted grid."""
    ax = _plt.gca()
    ax.set_xlabel(_x_label)
    ax.set_ylabel(_y_label)
    if _title:
        ax.set_title(_title)
    ax.grid(True, linestyle=":", color="gray", alpha=0.3)
    ax.set_axisbelow(True)
    if _legend_handles:
        ax.legend(handles=_legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1),
                  borderaxespad=0, frameon=False, fontsize=12)
    fig = ax.figure
    if _legend_handles:
        fig.tight_layout(rect=(0, 0, 0.82, 1))
    else:
        fig.tight_layout()
    file_path = str(_file_path)
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    for fmt in plot_formats:
        try:
            fig.savefig(file_path + "." + fmt, format=fmt if fmt != "png" else None,
                        dpi=300 if fmt == "png" else None, bbox_inches="tight", pad_inches=0.1)
        except Exception as exc:
            if fmt == "pgf":
                warnings.warn("PGF export failed for {}: {}".format(file_path, exc), RuntimeWarning)
            else:
                raise
    _plt.close(fig)

# Scenario A has no Drift 2 clients before its second change point. Represent it as a gap.
_original_batch_stats = per_round_stats_for_clients_in_range
def per_round_stats_for_clients_in_range(client_log, clients, start_t, end_t):
    if clients:
        return _original_batch_stats(client_log, clients, start_t, end_t)
    empty = float("nan")
    return {t: {"loss_min": empty, "loss_max": empty, "loss_avg": empty,
                "acc_min": empty, "acc_max": empty, "acc_avg": empty}
            for t in range(start_t, end_t)}


def _batch_smooth_log(client_log, window=6):
    """Apply a centered moving average per client to loss and accuracy."""
    left, right = (window - 1) // 2, window // 2
    smoothed = []
    for t in range(len(client_log)):
        start, end = max(0, t - left), min(len(client_log), t + right + 1)
        smoothed.append([
            (float(np.mean([client_log[k][c][0] for k in range(start, end)])),
             float(np.mean([client_log[k][c][1] for k in range(start, end)])))
            for c in range(len(client_log[0]))
        ])
    return smoothed


for scenario in "ABC":
    scenario_root = BATCH_LOG_ROOT / scenario / BATCH_DATASET
    drift_specs = read_logs(str(scenario_root / "saved_logs_fedavg" / "drift_specs_log.pkl"))
    drift_steps = drift_specs["drift_step_rounds"]
    drifted = drift_specs["drift_clustered_client_indices"]
    if len(drifted) != len(BATCH_PATTERNS[scenario]):
        raise ValueError("Drift-group metadata does not match Scenario {} pattern".format(scenario))
    reordered = reorder_clusters_by_pattern(drifted, BATCH_PATTERNS[scenario])
    if scenario == "A":
        reordered[0] = [reordered[0][0] + reordered[0][1], []]
    all_drifted = {client for step in reordered for group in step for client in group}
    client_count = len(read_logs(str(scenario_root / "saved_logs_fedavg" / "client_log.pkl"))[0])
    no_drift = sorted(set(range(client_count)) - all_drifted)
    clusters_per_segment = [step + [no_drift] for step in reordered]

    run_configs = []
    smooth_configs = []
    for method_name, folder, color in BATCH_METHODS:
        log_path = scenario_root / folder / "client_log.pkl"
        if not log_path.is_file():
            raise FileNotFoundError("Missing method log: {}".format(log_path))
        raw_log = read_logs(str(log_path))
        pre, post = build_plot_inputs_for_log(raw_log, drift_steps, clusters_per_segment)
        run_configs.append({"name": method_name, "pre_stats": pre,
                            "post_cluster_plot_stats": post, "color": color,
                            "cluster_linestyles": {0: "--", 1: "-.", 2: ":"},
                            "show_in_legend": True, "legend_mode": "method"})
        if BATCH_DATASET == "CIFAR-10":
            smooth_pre, smooth_post = build_plot_inputs_for_log(
                _batch_smooth_log(raw_log, window=6), drift_steps, clusters_per_segment)
            smooth_configs.append({"name": method_name, "pre_stats": smooth_pre,
                                   "post_cluster_plot_stats": smooth_post, "color": color,
                                   "cluster_linestyles": {0: "--", 1: "-.", 2: ":"},
                                   "show_in_legend": True, "legend_mode": "method"})

    annotation_names = {
        "A": [],
        "B": [r"$T_2$", r"$T_3$"],
        "C": [r"$T_4$", r"$T_5$", r"$T_6$"],
    }[scenario]
    if scenario != "A" and len(annotation_names) != len(drift_steps) - 1:
        raise ValueError("Scenario {} needs {} interval labels; got {} drift boundaries".format(
            scenario, len(drift_steps) - 1, len(annotation_names)))
    scenario_annotations = [
        (left, right, label)
        for left, right, label in zip(drift_steps[:-1], drift_steps[1:], annotation_names)
    ]

    output_root = BATCH_FIGURE_ROOT / scenario
    combined_prefix = output_root / ("1_clients_performance_" + BATCH_DATASET.replace("-", "_"))
    plot_multiple_methods_pre_post_drift(
        method_configs=run_configs, drift_steps=drift_steps, out_prefix=str(combined_prefix),
        band_alpha=0.25, show_band=False, stitch=True)
    cluster_prefix = output_root / ("3_clusterwise_comparison_" + BATCH_DATASET)
    plot_all_clusters_separate(
        method_configs=run_configs, drift_steps=drift_steps,
        cluster_map={0: "drift_1", 1: "drift_2", 2: "no_drift"},
        drift_type_linestyles={"drift_1": "--", "drift_2": "-.", "no_drift": ":"},
        out_prefix=str(cluster_prefix), band_alpha=0.25, show_band=False,
        stitch=True, smooth_window=1, smooth_mode="moving_avg",
        t1_range=drift_steps[:2] if scenario == "A" else None,
        period_annotations=scenario_annotations, scenario=scenario)
    if BATCH_DATASET == "CIFAR-10":
        smooth_root = output_root / "smooth"
        smooth_combined_prefix = smooth_root / ("1_clients_performance_" + BATCH_DATASET.replace("-", "_"))
        plot_multiple_methods_pre_post_drift(
            method_configs=smooth_configs, drift_steps=drift_steps,
            out_prefix=str(smooth_combined_prefix), band_alpha=0.25,
            show_band=False, stitch=True)
        smooth_cluster_prefix = smooth_root / ("3_clusterwise_comparison_" + BATCH_DATASET)
        plot_all_clusters_separate(
            method_configs=smooth_configs, drift_steps=drift_steps,
            cluster_map={0: "drift_1", 1: "drift_2", 2: "no_drift"},
            drift_type_linestyles={"drift_1": "--", "drift_2": "-.", "no_drift": ":"},
            out_prefix=str(smooth_cluster_prefix), band_alpha=0.25,
            show_band=False, stitch=True, smooth_window=1, smooth_mode="moving_avg",
            t1_range=drift_steps[:2] if scenario == "A" else None,
            period_annotations=scenario_annotations, scenario=scenario)
    print("Finished", BATCH_DATASET, "Scenario", scenario, "->", output_root)


### Scenario A drift_1 figures with T1 interval annotation


In [ ]:
# Scenario A drift_1 figures with the T1 interval annotation.
from pathlib import Path
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from log_utils.logging import read_logs

T1_DATASET = "CIFAR-10"
T1_LOG_ROOT = Path("D:/publications_related/2026/ICDCS/logs/swap/seed42/A") / T1_DATASET
T1_OUTPUT_ROOT = Path("D:/publications_related/2026/ICDCS/figures/new/seed42/l3/A")
T1_METHODS = [
    ("FedAvg", "saved_logs_fedavg", "red"),
    ("Oracle", "saved_logs_oracle", "green"),
    ("FairFedDrift", "saved_logs_fairfeddrift", "darkviolet"),
    ("FedBABU", "saved_logs_fedbabu", "darkorange"),
    ("PaLA", "saved_logs_fedex", "blue"),
]
T1_DRIFT_RANGE = None

plt.rcParams.update({"savefig.bbox": "tight", "savefig.pad_inches": 0.1,
                     "font.size": 20, "axes.labelsize": 20,
                     "xtick.labelsize": 20, "ytick.labelsize": 20})

def configure_and_save_plot(_plt, _x_label, _y_label, _title, _file_path,
                            _legend_handles=None, plot_formats=("png", "pdf", "pgf")):
    """Save a title-free drift_1 plot with a T1 double arrow and no variance band."""
    ax = _plt.gca()
    ax.set_xlabel(_x_label)
    ax.set_ylabel(_y_label)
    ax.grid(True, linestyle=":", color="gray", alpha=0.3)
    ax.set_axisbelow(True)
    ax.margins(x=0.05)

    left, right = T1_DRIFT_RANGE
    transform = ax.get_xaxis_transform()
    ax.annotate("", xy=(right, 0.16), xytext=(left, 0.16),
                xycoords=transform, textcoords=transform,
                arrowprops={"arrowstyle": "<->", "color": "black", "linewidth": 1.2,
                            "mutation_scale": 12, "shrinkA": 0, "shrinkB": 0},
                zorder=6)
    ax.text((left + right) / 2, 0.12, r"$T_1$", transform=transform,
            ha="center", va="top", fontsize=24, color="black", zorder=6)

    fig = ax.figure
    fig.tight_layout()
    file_path = str(_file_path)
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    for fmt in plot_formats:
        try:
            fig.savefig(file_path + "." + fmt, format=fmt if fmt != "png" else None,
                        dpi=300 if fmt == "png" else None, bbox_inches="tight", pad_inches=0.1)
        except Exception as exc:
            if fmt == "pgf":
                warnings.warn("PGF export failed for {}: {}".format(file_path, exc), RuntimeWarning)
            else:
                raise
    _plt.close(fig)


def _t1_smooth_log(client_log, window=6):
    """Apply a centered moving average per client to loss and accuracy."""
    left, right = (window - 1) // 2, window // 2
    output = []
    for t in range(len(client_log)):
        start, end = max(0, t - left), min(len(client_log), t + right + 1)
        output.append([
            (float(np.mean([client_log[k][c][0] for k in range(start, end)])),
             float(np.mean([client_log[k][c][1] for k in range(start, end)])))
            for c in range(len(client_log[0]))
        ])
    return output


_t1_original_stats = per_round_stats_for_clients_in_range
def per_round_stats_for_clients_in_range(client_log, clients, start_t, end_t):
    if clients:
        return _t1_original_stats(client_log, clients, start_t, end_t)
    _empty = float("nan")
    return {t: {"loss_min": _empty, "loss_max": _empty, "loss_avg": _empty,
                "acc_min": _empty, "acc_max": _empty, "acc_avg": _empty}
            for t in range(start_t, end_t)}

_t1_specs = read_logs(str(T1_LOG_ROOT / "saved_logs_fedavg" / "drift_specs_log.pkl"))
_t1_steps = _t1_specs["drift_step_rounds"]
if len(_t1_steps) < 2:
    raise ValueError("Scenario A needs two drift boundaries to draw T1")
T1_DRIFT_RANGE = (_t1_steps[0], _t1_steps[1])
_t1_groups = reorder_clusters_by_pattern(
    _t1_specs["drift_clustered_client_indices"], [[1], [1, 2]])
_t1_groups[0] = [_t1_groups[0][0] + _t1_groups[0][1], []]
_t1_reference_log = read_logs(str(T1_LOG_ROOT / "saved_logs_fedavg" / "client_log.pkl"))
_t1_drifted = {client for step in _t1_groups for group in step for client in group}
_t1_no_drift = sorted(set(range(len(_t1_reference_log[0]))) - _t1_drifted)
_t1_clusters = [step + [_t1_no_drift] for step in _t1_groups]

_t1_raw_configs = []
_t1_smooth_configs = []
for _method, _folder, _color in T1_METHODS:
    _log = read_logs(str(T1_LOG_ROOT / _folder / "client_log.pkl"))
    if T1_DATASET in ("MNIST", "F_MNIST") and len(_log) != 50:
        raise ValueError("Expected 50 rounds for {}; found {}".format(_folder, len(_log)))
    _pre, _post = build_plot_inputs_for_log(_log, _t1_steps, _t1_clusters)
    _t1_raw_configs.append({
        "name": _method, "pre_stats": _pre, "post_cluster_plot_stats": _post,
        "color": _color, "cluster_linestyles": {0: "--", 1: "-.", 2: ":"},
        "show_in_legend": False, "legend_mode": "none",
    })
    if T1_DATASET == "CIFAR-10":
        _smooth_pre, _smooth_post = build_plot_inputs_for_log(
            _t1_smooth_log(_log, window=6), _t1_steps, _t1_clusters)
        _t1_smooth_configs.append({
            "name": _method, "pre_stats": _smooth_pre,
            "post_cluster_plot_stats": _smooth_post, "color": _color,
            "cluster_linestyles": {0: "--", 1: "-.", 2: ":"},
            "show_in_legend": False, "legend_mode": "none",
        })

_t1_versions = [(T1_OUTPUT_ROOT, _t1_raw_configs)]
if T1_DATASET == "CIFAR-10":
    _t1_versions.append((T1_OUTPUT_ROOT / "smooth", _t1_smooth_configs))
for _version_root, _configs in _t1_versions:
    for _metric, _ylabel in (("acc", "Accuracy"), ("loss", "Loss")):
        _scenario = "A"
        _file_prefix = _version_root / "3_clusterwise_comparison_{}_{}_drift_1_{}".format(
            T1_DATASET, _scenario, "accuracy" if _metric == "acc" else "loss")
        plot_one_cluster_across_methods(
            method_configs=_configs, cluster_idx=0, metric_prefix=_metric,
            drift_steps=_t1_steps, out_file_path=str(_file_prefix),
            post_linestyle="--", title="", band_alpha=0.25,
            show_band=False, stitch=True, smooth_window=1, smooth_mode="moving_avg")
print("Generated Scenario A drift_1 T1 figures for", T1_DATASET)


In [ ]:
# Export the same raw Scenario B rotation figures without y-axis labels.
rotation_no_ylabel_prefix = str(ROTATION_FIGURE_ROOT / "1_clients_performance_CIFAR-10_rotation")
plot_multiple_methods_pre_post_drift(
    method_configs=rotation_raw_configs, drift_steps=drift_steps,
    out_prefix=rotation_no_ylabel_prefix, band_alpha=0.25, show_band=True,
    stitch=True, show_y_label=False, filename_suffix="_wo_ax_label", period_annotations=rotation_period_annotations,
)
plot_all_clusters_separate(
    method_configs=rotation_raw_configs, drift_steps=drift_steps,
    cluster_map=rotation_cluster_map, drift_type_linestyles=rotation_drift_styles,
    out_prefix=str(ROTATION_FIGURE_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=True, stitch=True, show_y_label=False,
    filename_suffix="_wo_ax_label", period_annotations=rotation_period_annotations,
)
print("Saved raw rotation figures without y-axis labels in:", ROTATION_FIGURE_ROOT)



In [ ]:
# Export matching nine-round-smoothed rotation figures without y-axis labels.
rotation_smooth_no_ylabel_prefix = str(ROTATION_SMOOTH_ROOT / "1_clients_performance_CIFAR-10_rotation")
plot_multiple_methods_pre_post_drift(
    method_configs=rotation_smooth_configs, drift_steps=drift_steps,
    out_prefix=rotation_smooth_no_ylabel_prefix, band_alpha=0.25, show_band=True,
    stitch=True, show_y_label=False, filename_suffix="_wo_ax_label", period_annotations=rotation_period_annotations,
)
plot_all_clusters_separate(
    method_configs=rotation_smooth_configs, drift_steps=drift_steps,
    cluster_map=rotation_cluster_map, drift_type_linestyles=rotation_drift_styles,
    out_prefix=str(ROTATION_SMOOTH_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=True, stitch=True, show_y_label=False,
    filename_suffix="_wo_ax_label", period_annotations=rotation_period_annotations,
)
print("Saved smoothed rotation figures without y-axis labels in:", ROTATION_SMOOTH_ROOT)


In [ ]:
# Raw rotation plots with average curves only (no standard-deviation bands).
plot_multiple_methods_pre_post_drift(
    method_configs=rotation_raw_configs, drift_steps=drift_steps,
    out_prefix=str(ROTATION_FIGURE_ROOT / "1_clients_performance_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=False, stitch=True, show_y_label=False,
    filename_suffix="_wo_ax_label_nov", period_annotations=rotation_period_annotations,
)
plot_all_clusters_separate(
    method_configs=rotation_raw_configs, drift_steps=drift_steps,
    cluster_map=rotation_cluster_map, drift_type_linestyles=rotation_drift_styles,
    out_prefix=str(ROTATION_FIGURE_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=False, stitch=True, show_y_label=False,
    filename_suffix="_wo_ax_label_nov", period_annotations=rotation_period_annotations,
)
print("Saved raw average-only rotation figures in:", ROTATION_FIGURE_ROOT)


In [ ]:
# Nine-round-smoothed rotation plots with average curves only (no standard-deviation bands).
plot_multiple_methods_pre_post_drift(
    method_configs=rotation_smooth_configs, drift_steps=drift_steps,
    out_prefix=str(ROTATION_SMOOTH_ROOT / "1_clients_performance_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=False, stitch=True, show_y_label=False,
    filename_suffix="_wo_ax_label_nov", period_annotations=rotation_period_annotations,
)
plot_all_clusters_separate(
    method_configs=rotation_smooth_configs, drift_steps=drift_steps,
    cluster_map=rotation_cluster_map, drift_type_linestyles=rotation_drift_styles,
    out_prefix=str(ROTATION_SMOOTH_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=False, stitch=True, show_y_label=False,
    filename_suffix="_wo_ax_label_nov", period_annotations=rotation_period_annotations,
)
print("Saved smoothed average-only rotation figures in:", ROTATION_SMOOTH_ROOT)


In [ ]:
# Raw average-only rotation plots with the y-axis labels retained.
plot_multiple_methods_pre_post_drift(
    method_configs=rotation_raw_configs, drift_steps=drift_steps,
    out_prefix=str(ROTATION_FIGURE_ROOT / "1_clients_performance_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=False, stitch=True, show_y_label=True,
    filename_suffix="_nov", period_annotations=rotation_period_annotations,
)
plot_all_clusters_separate(
    method_configs=rotation_raw_configs, drift_steps=drift_steps,
    cluster_map=rotation_cluster_map, drift_type_linestyles=rotation_drift_styles,
    out_prefix=str(ROTATION_FIGURE_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=False, stitch=True, show_y_label=True,
    filename_suffix="_nov", period_annotations=rotation_period_annotations,
)
print("Saved labeled raw average-only rotation figures in:", ROTATION_FIGURE_ROOT)


In [ ]:
# Nine-round-smoothed average-only rotation plots with the y-axis labels retained.
plot_multiple_methods_pre_post_drift(
    method_configs=rotation_smooth_configs, drift_steps=drift_steps,
    out_prefix=str(ROTATION_SMOOTH_ROOT / "1_clients_performance_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=False, stitch=True, show_y_label=True,
    filename_suffix="_nov", period_annotations=rotation_period_annotations,
)
plot_all_clusters_separate(
    method_configs=rotation_smooth_configs, drift_steps=drift_steps,
    cluster_map=rotation_cluster_map, drift_type_linestyles=rotation_drift_styles,
    out_prefix=str(ROTATION_SMOOTH_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation"),
    band_alpha=0.25, show_band=False, stitch=True, show_y_label=True,
    filename_suffix="_nov", period_annotations=rotation_period_annotations,
)
print("Saved labeled smoothed average-only rotation figures in:", ROTATION_SMOOTH_ROOT)


In [ ]:
# Regenerate only the requested smoothed rotation average-only accuracy plots at L-ablation line width.
ROTATION_FINAL_ROOT = ROTATION_SMOOTH_ROOT / "final"
ROTATION_FINAL_ROOT.mkdir(parents=True, exist_ok=True)
plot_one_cluster_across_methods(
    method_configs=rotation_smooth_configs, cluster_idx=0, metric_prefix="acc",
    drift_steps=drift_steps,
    out_file_path=str(ROTATION_FINAL_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation_drift_1_accuracy"),
    post_linestyle=rotation_drift_styles["drift_1"], band_alpha=0.25,
    show_band=False, stitch=True, smooth_window=1,
    period_annotations=rotation_period_annotations, show_y_label=True,
    filename_suffix="_nov", pre_linewidth=2.7, post_linewidth=2.7,
)
plot_one_cluster_across_methods(
    method_configs=rotation_smooth_configs, cluster_idx=1, metric_prefix="acc",
    drift_steps=drift_steps,
    out_file_path=str(ROTATION_FINAL_ROOT / "3_clusterwise_comparison_CIFAR-10_rotation_drift_2_accuracy"),
    post_linestyle=rotation_drift_styles["drift_2"], band_alpha=0.25,
    show_band=False, stitch=True, smooth_window=1,
    period_annotations=rotation_period_annotations, show_y_label=False,
    filename_suffix="_wo_ax_label_nov", pre_linewidth=2.7, post_linewidth=2.7,
)
print("Saved the two thicker smoothed rotation accuracy plots in:", ROTATION_FINAL_ROOT)


In [ ]:
# Regenerate only the smoothed CIFAR-10 rotation legend.
from pathlib import Path
from matplotlib.lines import Line2D
import os
import warnings

_ROTATION_LEGEND_PREFIX = Path(
    "D:/publications_related/2026/ICDCS/figures/new/seed42/B/"
    "CIFAR-10_rotation/smooth/CIFAR-10_rotation_legend"
)
_ROTATION_LEGEND_HANDLES = [
    Line2D([0], [0], color=cfg["color"], linewidth=3.6, alpha=0.6,
           label=cfg["name"])
    for cfg in rotation_smooth_configs
]
_ROTATION_LEGEND_FIG = plt.figure(
    figsize=(max(6, len(_ROTATION_LEGEND_HANDLES) * 1.4), 0.8)
)
_ROTATION_LEGEND_FIG.legend(
    handles=_ROTATION_LEGEND_HANDLES,
    labels=[cfg["name"] for cfg in rotation_smooth_configs], loc="center",
    ncol=len(_ROTATION_LEGEND_HANDLES), frameon=False, fontsize=16,
    handlelength=2.5, columnspacing=1.5,
)
os.makedirs(str(_ROTATION_LEGEND_PREFIX.parent), exist_ok=True)
_ROTATION_LEGEND_FIG.savefig(str(_ROTATION_LEGEND_PREFIX) + ".png", dpi=300,
                             bbox_inches="tight", pad_inches=0.1)
_ROTATION_LEGEND_FIG.savefig(str(_ROTATION_LEGEND_PREFIX) + ".pdf",
                             bbox_inches="tight", pad_inches=0.1)
try:
    _ROTATION_LEGEND_FIG.savefig(
        str(_ROTATION_LEGEND_PREFIX) + ".pgf", format="pgf",
        bbox_inches="tight", pad_inches=0.1,
    )
except Exception as exc:
    warnings.warn("PGF legend export failed: {}".format(exc), RuntimeWarning)
plt.close(_ROTATION_LEGEND_FIG)
print("Saved corrected smoothed rotation legend:", _ROTATION_LEGEND_PREFIX)


In [ ]:
# Independent CIFAR-10 batch plots for seed51 and seed52, Scenarios A/B/C.
# Reads the standard FedAvg, Oracle, FairFedDrift, FedBABU, and PaLA logs.
from pathlib import Path
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from log_utils.logging import read_logs

MULTISEED_LOG_ROOT = Path("D:/publications_related/2026/ICDCS/logs/swap")
MULTISEED_FIGURE_ROOT = Path("D:/publications_related/2026/ICDCS/figures/new")
MULTISEED_SEEDS = ("seed51", "seed52")
MULTISEED_DATASET = "CIFAR-10"
MULTISEED_METHODS = [
    ("FedAvg", "saved_logs_fedavg", "red"),
    ("Oracle", "saved_logs_oracle", "green"),
    ("FairFedDrift", "saved_logs_fairfeddrift", "darkviolet"),
    ("FedBABU", "saved_logs_fedbabu", "darkorange"),
    ("PaLA", "saved_logs_fedex", "blue"),
]
MULTISEED_PATTERNS = {
    "A": [[1], [1, 2]],
    "B": [[1, 2], [1, 2]],
    "C": [[1, 2], [1, 2], [2, 1]],
}

plt.rcParams.update({"savefig.bbox": "tight", "savefig.pad_inches": 0.1,
                     "font.size": 20, "axes.labelsize": 20,
                     "xtick.labelsize": 20, "ytick.labelsize": 20})

def configure_and_save_plot(_plt, _x_label, _y_label, _title, _file_path,
                            _legend_handles=None, plot_formats=("png", "pdf", "pgf")):
    """Format and save a plot with dotted grid lines."""
    ax = _plt.gca()
    ax.set_xlabel(_x_label)
    ax.set_ylabel(_y_label)
    if _title:
        ax.set_title(_title)
    ax.grid(True, linestyle=":", color="gray", alpha=0.3)
    ax.set_axisbelow(True)
    if _legend_handles:
        ax.legend(handles=_legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1),
                  borderaxespad=0, frameon=False, fontsize=12)
    fig = ax.figure
    fig.tight_layout(rect=(0, 0, 0.82, 1) if _legend_handles else None)
    file_path = str(_file_path)
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    for fmt in plot_formats:
        try:
            fig.savefig(file_path + "." + fmt, format=fmt if fmt != "png" else None,
                        dpi=300 if fmt == "png" else None, bbox_inches="tight", pad_inches=0.1)
        except Exception as exc:
            if fmt == "pgf":
                warnings.warn("PGF export failed for {}: {}".format(file_path, exc), RuntimeWarning)
            else:
                raise
    _plt.close(fig)

# Scenario A has no drift-2 clients before its second drift boundary; keep that range blank.
_multiseed_original_stats = per_round_stats_for_clients_in_range
def per_round_stats_for_clients_in_range(client_log, clients, start_t, end_t):
    if clients:
        return _multiseed_original_stats(client_log, clients, start_t, end_t)
    empty = float("nan")
    return {t: {"loss_min": empty, "loss_max": empty, "loss_avg": empty,
                "acc_min": empty, "acc_max": empty, "acc_avg": empty}
            for t in range(start_t, end_t)}

def _multiseed_smooth_log(client_log, window=6):
    """Apply a centered six-round moving average per client."""
    left, right = (window - 1) // 2, window // 2
    return [
        [(float(np.mean([client_log[k][c][0] for k in range(max(0, t-left), min(len(client_log), t+right+1))])),
          float(np.mean([client_log[k][c][1] for k in range(max(0, t-left), min(len(client_log), t+right+1))])))
         for c in range(len(client_log[0]))]
        for t in range(len(client_log))
    ]

def _multiseed_add_interval_arrows(ax, intervals):
    """Draw the notebook-style double arrows and interval labels."""
    transform = ax.get_xaxis_transform()
    for left, right, label in intervals:
        ax.annotate("", xy=(right, 0.16), xytext=(left, 0.16),
                    xycoords=transform, textcoords=transform,
                    arrowprops={"arrowstyle": "<->", "color": "black", "linewidth": 1.2,
                                "mutation_scale": 12, "shrinkA": 0, "shrinkB": 0}, zorder=6)
        ax.text((left + right) / 2, 0.12, label, transform=transform,
                ha="center", va="top", fontsize=24, color="black", zorder=6)

# The helper below mirrors the separate-cluster plotting function while adding
# the correct scenario intervals to every applicable plot, including Scenario A T1.
def _multiseed_plot_clusters(method_configs, drift_steps, out_prefix, scenario, intervals,
                             smooth=False):
    cluster_map = {0: "drift_1", 1: "drift_2", 2: "no_drift"}
    drift_styles = {"drift_1": "--", "drift_2": "-.", "no_drift": ":"}
    for cluster_idx, drift_name in cluster_map.items():
        post_style = drift_styles[drift_name]
        file_stem = ("{}_{}_drift_1".format(out_prefix, scenario)
                     if drift_name == "drift_1" else "{}_{}".format(out_prefix, drift_name))
        for metric, ylabel in (("acc", "Accuracy"), ("loss", "Loss")):
            fig, ax = plt.subplots()
            for cfg in method_configs:
                plot_pre_post_single_cluster_on_axis(
                    ax, pre_stats=cfg["pre_stats"],
                    post_cluster_plot_stats=cfg["post_cluster_plot_stats"],
                    cluster_idx=cluster_idx, metric_prefix=metric,
                    base_color=cfg["color"], method_label=cfg["name"],
                    pre_linestyle="-", pre_linewidth=1.8, post_linestyle=post_style,
                    post_linewidth=1.6, band_alpha=0.25, show_band=False,
                    stitch=True, drift_steps=sorted(drift_steps), collect_legend=False)
            _multiseed_add_interval_arrows(ax, intervals)
            configure_and_save_plot(
                plt, "Round", ylabel, "", file_stem + "_" + ("accuracy" if metric == "acc" else "loss"))
    version = "smoothed" if smooth else "raw"
    print("  Wrote", version, "combined and clusterwise plots to", Path(out_prefix).parent)

for _seed in MULTISEED_SEEDS:
    for _scenario in "ABC":
        _scenario_root = MULTISEED_LOG_ROOT / _seed / _scenario / MULTISEED_DATASET
        _reference = _scenario_root / "saved_logs_fedavg"
        _specs = read_logs(str(_reference / "drift_specs_log.pkl"))
        _steps = _specs["drift_step_rounds"]
        _groups = reorder_clusters_by_pattern(
            _specs["drift_clustered_client_indices"], MULTISEED_PATTERNS[_scenario])
        if _scenario == "A":
            _groups[0] = [_groups[0][0] + _groups[0][1], []]
        _all_drifted = {client for step in _groups for group in step for client in group}
        _client_count = len(read_logs(str(_reference / "client_log.pkl"))[0])
        _no_drift = sorted(set(range(_client_count)) - _all_drifted)
        _clusters = [step + [_no_drift] for step in _groups]
        _raw_configs, _smooth_configs = [], []
        for _name, _folder, _color in MULTISEED_METHODS:
            _log_path = _scenario_root / _folder / "client_log.pkl"
            if not _log_path.is_file():
                raise FileNotFoundError("Missing method log: {}".format(_log_path))
            _log = read_logs(str(_log_path))
            if len(_log) != 400:
                raise ValueError("Expected 400 CIFAR-10 rounds in {}; found {}".format(_log_path, len(_log)))
            _pre, _post = build_plot_inputs_for_log(_log, _steps, _clusters)
            _raw_configs.append({"name": _name, "pre_stats": _pre,
                                 "post_cluster_plot_stats": _post, "color": _color})
            _s_pre, _s_post = build_plot_inputs_for_log(
                _multiseed_smooth_log(_log, window=6), _steps, _clusters)
            _smooth_configs.append({"name": _name, "pre_stats": _s_pre,
                                    "post_cluster_plot_stats": _s_post, "color": _color})
        _labels = {"A": [r"$T_1$"], "B": [r"$T_2$", r"$T_3$"],
                   "C": [r"$T_4$", r"$T_5$", r"$T_6$"]}[_scenario]
        _interval_bounds = list(zip(_steps[:-1], _steps[1:]))
        _interval_bounds = _interval_bounds[:1] if _scenario == "A" else _interval_bounds
        _intervals = [(left, right, label) for (left, right), label in
                      zip(_interval_bounds, _labels)]
        if len(_intervals) != len(_labels):
            raise ValueError("Scenario {} drift boundaries do not match interval labels".format(_scenario))
        _out = MULTISEED_FIGURE_ROOT / _seed / _scenario
        _out.mkdir(parents=True, exist_ok=True)
        for _kind, _configs in (("raw", _raw_configs), ("smooth", _smooth_configs)):
            _version_root = _out if _kind == "raw" else _out / "smooth"
            _version_root.mkdir(parents=True, exist_ok=True)
            _combined = str(_version_root / ("1_clients_performance_CIFAR_10"))
            plot_multiple_methods_pre_post_drift(
                method_configs=_configs, drift_steps=_steps, out_prefix=_combined,
                band_alpha=0.25, show_band=False, stitch=True, period_annotations=_intervals)
            _cluster_prefix = str(_version_root / "3_clusterwise_comparison_CIFAR-10")
            _multiseed_plot_clusters(_configs, _steps, _cluster_prefix, _scenario,
                                     _intervals, smooth=(_kind == "smooth"))
        print("Finished", _seed, "Scenario", _scenario, "->", _out)


In [ ]:
# Create plotting logs by averaging seeds 42, 51, and 52 for CIFAR-10.
from pathlib import Path
import pickle
import numpy as np
from log_utils.logging import read_logs

AVG_LOG_ROOT = Path(r"D:\publications_related\2026\ICDCS\logs\swap")
AVG_DATASET = 'CIFAR-10'
AVG_SEEDS = ("seed42", "seed51", "seed52")
AVG_METHODS = {
    "fedavg": "fedavg", "oracle": "oracle", "fairfeddrift": "fairfeddrift",
    "fedbabu": "fedbabu", "pala": "fedex",
}

for scenario in "ABC":
    reference_specs = None
    method_logs = {}
    for label, folder in AVG_METHODS.items():
        seed_logs = []
        seed_specs = []
        for seed in AVG_SEEDS:
            log_dir = AVG_LOG_ROOT / seed / scenario / AVG_DATASET / f"saved_logs_{folder}"
            seed_logs.append(read_logs(str(log_dir / "client_log.pkl")))
            seed_specs.append(read_logs(str(log_dir / "drift_specs_log.pkl")))
        if not all(spec == seed_specs[0] for spec in seed_specs):
            raise ValueError(f"Drift metadata differs across seeds for {AVG_DATASET}/{scenario}/{label}")
        if reference_specs is None:
            reference_specs = seed_specs[0]
        elif seed_specs[0] != reference_specs:
            raise ValueError(f"Drift metadata differs across methods for {AVG_DATASET}/{scenario}/{label}")
        shape = (len(seed_logs[0]), len(seed_logs[0][0]))
        if any((len(log), len(log[0])) != shape for log in seed_logs):
            raise ValueError(f"Log dimensions differ across seeds for {AVG_DATASET}/{scenario}/{label}")
        method_logs[label] = seed_logs

    for label, seed_logs in method_logs.items():
        averaged_log = []
        for round_index in range(len(seed_logs[0])):
            averaged_round = []
            for client_index in range(len(seed_logs[0][round_index])):
                averaged_round.append((
                    float(np.mean([log[round_index][client_index][0] for log in seed_logs])),
                    float(np.mean([log[round_index][client_index][1] for log in seed_logs])),
                ))
            averaged_log.append(averaged_round)
        folder = AVG_METHODS[label]
        output_dir = AVG_LOG_ROOT / "seed_avg" / scenario / AVG_DATASET / f"saved_logs_{folder}"
        output_dir.mkdir(parents=True, exist_ok=True)
        with (output_dir / "client_log.pkl").open("wb") as output_file:
            pickle.dump(averaged_log, output_file)
        with (output_dir / "drift_specs_log.pkl").open("wb") as output_file:
            pickle.dump(reference_specs, output_file)
    print("Saved averaged plotting logs:", AVG_DATASET, "Scenario", scenario)


In [ ]:
# Plots from three-seed averaged logs for CIFAR-10, Scenarios A/B/C.
from pathlib import Path
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from log_utils.logging import read_logs

BATCH_LOG_ROOT = Path("D:/publications_related/2026/ICDCS/logs/swap/seed_avg")
BATCH_FIGURE_ROOT = Path("D:/publications_related/2026/ICDCS/figures/new/average/CIFAR-10")
BATCH_DATASET = "CIFAR-10"
BATCH_METHODS = [
    ("FedAvg", "saved_logs_fedavg", "red"),
    ("Oracle", "saved_logs_oracle", "green"),
    ("FairFedDrift", "saved_logs_fairfeddrift", "darkviolet"),
    ("FedBABU", "saved_logs_fedbabu", "darkorange"),
    ("PaLA", "saved_logs_fedex", "blue"),
]
BATCH_PATTERNS = {
    "A": [[1], [1, 2]],
    "B": [[1, 2], [1, 2]],
    "C": [[1, 2], [1, 2], [2, 1]],
}

plt.rcParams.update({"savefig.bbox": "tight", "savefig.pad_inches": 0.1,
                     "font.size": 20, "axes.labelsize": 20,
                     "xtick.labelsize": 20, "ytick.labelsize": 20})

def configure_and_save_plot(_plt, _x_label, _y_label, _title, _file_path,
                            _legend_handles=None, plot_formats=("png", "pdf", "pgf")):
    """Format and save a notebook plot with an external legend and dotted grid."""
    ax = _plt.gca()
    ax.set_xlabel(_x_label)
    ax.set_ylabel(_y_label)
    if _title:
        ax.set_title(_title)
    ax.grid(True, linestyle=":", color="gray", alpha=0.3)
    ax.set_axisbelow(True)
    if _legend_handles:
        ax.legend(handles=_legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1),
                  borderaxespad=0, frameon=False, fontsize=12)
    fig = ax.figure
    if _legend_handles:
        fig.tight_layout(rect=(0, 0, 0.82, 1))
    else:
        fig.tight_layout()
    file_path = str(_file_path)
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    for fmt in plot_formats:
        try:
            fig.savefig(file_path + "." + fmt, format=fmt if fmt != "png" else None,
                        dpi=300 if fmt == "png" else None, bbox_inches="tight", pad_inches=0.1)
        except Exception as exc:
            if fmt == "pgf":
                warnings.warn("PGF export failed for {}: {}".format(file_path, exc), RuntimeWarning)
            else:
                raise
    _plt.close(fig)

# Scenario A has no Drift 2 clients before its second change point. Represent it as a gap.
_original_batch_stats = per_round_stats_for_clients_in_range
def per_round_stats_for_clients_in_range(client_log, clients, start_t, end_t):
    if clients:
        return _original_batch_stats(client_log, clients, start_t, end_t)
    empty = float("nan")
    return {t: {"loss_min": empty, "loss_max": empty, "loss_avg": empty,
                "acc_min": empty, "acc_max": empty, "acc_avg": empty}
            for t in range(start_t, end_t)}


def _batch_smooth_log(client_log, window=6):
    """Apply a centered moving average per client to loss and accuracy."""
    left, right = (window - 1) // 2, window // 2
    smoothed = []
    for t in range(len(client_log)):
        start, end = max(0, t - left), min(len(client_log), t + right + 1)
        smoothed.append([
            (float(np.mean([client_log[k][c][0] for k in range(start, end)])),
             float(np.mean([client_log[k][c][1] for k in range(start, end)])))
            for c in range(len(client_log[0]))
        ])
    return smoothed


for scenario in "ABC":
    scenario_root = BATCH_LOG_ROOT / scenario / BATCH_DATASET
    drift_specs = read_logs(str(scenario_root / "saved_logs_fedavg" / "drift_specs_log.pkl"))
    drift_steps = drift_specs["drift_step_rounds"]
    drifted = drift_specs["drift_clustered_client_indices"]
    if len(drifted) != len(BATCH_PATTERNS[scenario]):
        raise ValueError("Drift-group metadata does not match Scenario {} pattern".format(scenario))
    reordered = reorder_clusters_by_pattern(drifted, BATCH_PATTERNS[scenario])
    if scenario == "A":
        reordered[0] = [reordered[0][0] + reordered[0][1], []]
    all_drifted = {client for step in reordered for group in step for client in group}
    client_count = len(read_logs(str(scenario_root / "saved_logs_fedavg" / "client_log.pkl"))[0])
    no_drift = sorted(set(range(client_count)) - all_drifted)
    clusters_per_segment = [step + [no_drift] for step in reordered]

    run_configs = []
    smooth_configs = []
    for method_name, folder, color in BATCH_METHODS:
        log_path = scenario_root / folder / "client_log.pkl"
        if not log_path.is_file():
            raise FileNotFoundError("Missing method log: {}".format(log_path))
        raw_log = read_logs(str(log_path))
        pre, post = build_plot_inputs_for_log(raw_log, drift_steps, clusters_per_segment)
        run_configs.append({"name": method_name, "pre_stats": pre,
                            "post_cluster_plot_stats": post, "color": color,
                            "cluster_linestyles": {0: "--", 1: "-.", 2: ":"},
                            "show_in_legend": True, "legend_mode": "method"})
        if BATCH_DATASET == "CIFAR-10":
            smooth_pre, smooth_post = build_plot_inputs_for_log(
                _batch_smooth_log(raw_log, window=6), drift_steps, clusters_per_segment)
            smooth_configs.append({"name": method_name, "pre_stats": smooth_pre,
                                   "post_cluster_plot_stats": smooth_post, "color": color,
                                   "cluster_linestyles": {0: "--", 1: "-.", 2: ":"},
                                   "show_in_legend": True, "legend_mode": "method"})

    annotation_names = {
        "A": [],
        "B": [r"$T_2$", r"$T_3$"],
        "C": [r"$T_4$", r"$T_5$", r"$T_6$"],
    }[scenario]
    if scenario != "A" and len(annotation_names) != len(drift_steps) - 1:
        raise ValueError("Scenario {} needs {} interval labels; got {} drift boundaries".format(
            scenario, len(drift_steps) - 1, len(annotation_names)))
    scenario_annotations = [
        (left, right, label)
        for left, right, label in zip(drift_steps[:-1], drift_steps[1:], annotation_names)
    ]

    output_root = BATCH_FIGURE_ROOT / scenario
    combined_prefix = output_root / ("1_clients_performance_" + BATCH_DATASET.replace("-", "_"))
    plot_multiple_methods_pre_post_drift(
        method_configs=run_configs, drift_steps=drift_steps, out_prefix=str(combined_prefix),
        band_alpha=0.25, show_band=False, stitch=True)
    cluster_prefix = output_root / ("3_clusterwise_comparison_" + BATCH_DATASET)
    plot_all_clusters_separate(
        method_configs=run_configs, drift_steps=drift_steps,
        cluster_map={0: "drift_1", 1: "drift_2", 2: "no_drift"},
        drift_type_linestyles={"drift_1": "--", "drift_2": "-.", "no_drift": ":"},
        out_prefix=str(cluster_prefix), band_alpha=0.25, show_band=False,
        stitch=True, smooth_window=1, smooth_mode="moving_avg",
        t1_range=drift_steps[:2] if scenario == "A" else None,
        period_annotations=scenario_annotations, scenario=scenario)
    if BATCH_DATASET == "CIFAR-10":
        smooth_root = output_root / "smooth"
        smooth_combined_prefix = smooth_root / ("1_clients_performance_" + BATCH_DATASET.replace("-", "_"))
        plot_multiple_methods_pre_post_drift(
            method_configs=smooth_configs, drift_steps=drift_steps,
            out_prefix=str(smooth_combined_prefix), band_alpha=0.25,
            show_band=False, stitch=True)
        smooth_cluster_prefix = smooth_root / ("3_clusterwise_comparison_" + BATCH_DATASET)
        plot_all_clusters_separate(
            method_configs=smooth_configs, drift_steps=drift_steps,
            cluster_map={0: "drift_1", 1: "drift_2", 2: "no_drift"},
            drift_type_linestyles={"drift_1": "--", "drift_2": "-.", "no_drift": ":"},
            out_prefix=str(smooth_cluster_prefix), band_alpha=0.25,
            show_band=False, stitch=True, smooth_window=1, smooth_mode="moving_avg",
            t1_range=drift_steps[:2] if scenario == "A" else None,
            period_annotations=scenario_annotations, scenario=scenario)
    print("Finished", BATCH_DATASET, "Scenario", scenario, "->", output_root)
